In [ ]:
# ---------------------------------------------------------
# API-FOOTBALL - STEP 1
# CONNECTION + PREMIER LEAGUE COVERAGE CHECK
# ---------------------------------------------------------

import os
import requests

from dotenv import load_dotenv


# ---------------------------------------------------------
# LOAD API KEY
# ---------------------------------------------------------

load_dotenv()


API_FOOTBALL_KEY = os.getenv(
    "API_FOOTBALL_KEY"
)


if not API_FOOTBALL_KEY:
    raise ValueError(
        "API_FOOTBALL_KEY was not found in your .env file."
    )


# ---------------------------------------------------------
# API SETTINGS
# ---------------------------------------------------------

BASE_URL = (
    "https://v3.football.api-sports.io"
)


HEADERS = {
    "x-apisports-key":
        API_FOOTBALL_KEY
}


PREMIER_LEAGUE_ID = 39


# API-Football represents seasons
# using the year in which the season begins.
SEASONS_TO_CHECK = [
    2020,
    2021,
    2022,
    2023,
    2024,
    2025,
    2026
]


# ---------------------------------------------------------
# BASIC CONNECTION TEST
# ---------------------------------------------------------

test_response = requests.get(
    f"{BASE_URL}/countries",
    headers=HEADERS,
    timeout=30
)


print(
    "Connection status:",
    test_response.status_code
)


test_data = (
    test_response.json()
)


print(
    "Connection errors:",
    test_data.get(
        "errors"
    )
)


print(
    "Countries returned:",
    test_data.get(
        "results"
    )
)


print()


# ---------------------------------------------------------
# CHECK PREMIER LEAGUE COVERAGE BY SEASON
# ---------------------------------------------------------

for season in SEASONS_TO_CHECK:

    response = requests.get(
        f"{BASE_URL}/leagues",
        headers=HEADERS,
        params={
            "id":
                PREMIER_LEAGUE_ID,

            "season":
                season
        },
        timeout=30
    )


    data = response.json()


    print(
        "=" * 60
    )

    print(
        "SEASON:",
        season
    )


    print(
        "HTTP STATUS:",
        response.status_code
    )


    print(
        "ERRORS:",
        data.get(
            "errors"
        )
    )


    print(
        "RESULTS:",
        data.get(
            "results"
        )
    )


    if data.get(
        "response"
    ):

        league_data = (
            data[
                "response"
            ][0]
        )


        print(
            "LEAGUE:",
            league_data[
                "league"
            ][
                "name"
            ]
        )


        for season_info in league_data.get(
            "seasons",
            []
        ):

            if (
                season_info.get(
                    "year"
                )
                ==
                season
            ):

                print(
                    "START:",
                    season_info.get(
                        "start"
                    )
                )

                print(
                    "END:",
                    season_info.get(
                        "end"
                    )
                )


                print(
                    "COVERAGE:"
                )


                print(
                    season_info.get(
                        "coverage"
                    )
                )


    print()

In [ ]:
# ---------------------------------------------------------
# API-FOOTBALL - STEP 2
# INSPECT ONE PREMIER LEAGUE MATCH
# ---------------------------------------------------------

import requests
import json


# ---------------------------------------------------------
# GET ONE FINISHED PREMIER LEAGUE MATCH
# FROM THE 2024/25 SEASON
# ---------------------------------------------------------

fixture_response = requests.get(
    f"{BASE_URL}/fixtures",
    headers=HEADERS,
    params={
        "league": PREMIER_LEAGUE_ID,
        "season": 2024,
        "from": "2024-08-16",
        "to": "2024-08-17"
    },
    timeout=30
)


fixture_data = fixture_response.json()


print(
    "FIXTURE REQUEST ERRORS:",
    fixture_data.get("errors")
)

print(
    "FIXTURES RETURNED:",
    fixture_data.get("results")
)


if not fixture_data.get("response"):
    raise ValueError(
        "No fixtures were returned."
    )


# ---------------------------------------------------------
# SELECT FIRST FIXTURE
# ---------------------------------------------------------

sample_fixture = (
    fixture_data["response"][0]
)


fixture_id = (
    sample_fixture["fixture"]["id"]
)


home_team = (
    sample_fixture["teams"]["home"]["name"]
)


away_team = (
    sample_fixture["teams"]["away"]["name"]
)


print()
print("SAMPLE MATCH:")
print(home_team, "vs", away_team)

print(
    "Fixture ID:",
    fixture_id
)


# ---------------------------------------------------------
# LINEUPS
# ---------------------------------------------------------

lineups_response = requests.get(
    f"{BASE_URL}/fixtures/lineups",
    headers=HEADERS,
    params={
        "fixture": fixture_id
    },
    timeout=30
)


lineups_data = (
    lineups_response.json()
)


print()
print("=" * 70)
print("LINEUPS")
print("=" * 70)

print(
    "Errors:",
    lineups_data.get("errors")
)

print(
    "Results:",
    lineups_data.get("results")
)


if lineups_data.get("response"):

    print(
        json.dumps(
            lineups_data["response"][0],
            indent=2
        )[:6000]
    )


# ---------------------------------------------------------
# PLAYER MATCH STATISTICS
# ---------------------------------------------------------

players_response = requests.get(
    f"{BASE_URL}/fixtures/players",
    headers=HEADERS,
    params={
        "fixture": fixture_id
    },
    timeout=30
)


players_data = (
    players_response.json()
)


print()
print("=" * 70)
print("PLAYER MATCH STATISTICS")
print("=" * 70)

print(
    "Errors:",
    players_data.get("errors")
)

print(
    "Results:",
    players_data.get("results")
)


if players_data.get("response"):

    print(
        json.dumps(
            players_data["response"][0],
            indent=2
        )[:8000]
    )


# ---------------------------------------------------------
# INJURIES
# ---------------------------------------------------------

injuries_response = requests.get(
    f"{BASE_URL}/injuries",
    headers=HEADERS,
    params={
        "fixture": fixture_id
    },
    timeout=30
)


injuries_data = (
    injuries_response.json()
)


print()
print("=" * 70)
print("INJURIES")
print("=" * 70)

print(
    "Errors:",
    injuries_data.get("errors")
)

print(
    "Results:",
    injuries_data.get("results")
)


if injuries_data.get("response"):

    print(
        json.dumps(
            injuries_data["response"],
            indent=2
        )[:8000]
    )


# ---------------------------------------------------------
# API QUOTA INFORMATION
# ---------------------------------------------------------

print()
print("=" * 70)
print("API QUOTA")
print("=" * 70)


print(
    "Requests remaining:",
    injuries_response.headers.get(
        "x-ratelimit-requests-remaining"
    )
)


print(
    "Daily request limit:",
    injuries_response.headers.get(
        "x-ratelimit-requests-limit"
    )
)

In [ ]:
# ---------------------------------------------------------
# API-FOOTBALL - STEP 3
# TEST SEASON-LEVEL INJURY + PLAYER DATA
# ---------------------------------------------------------

import requests
import json


TEST_SEASON = 2024


# ---------------------------------------------------------
# 1. SEASON-LEVEL INJURIES
# ---------------------------------------------------------

injuries_season_response = requests.get(
    f"{BASE_URL}/injuries",
    headers=HEADERS,
    params={
        "league": PREMIER_LEAGUE_ID,
        "season": TEST_SEASON
    },
    timeout=30
)


injuries_season_data = (
    injuries_season_response.json()
)


print("=" * 70)
print("SEASON INJURIES")
print("=" * 70)

print(
    "Errors:",
    injuries_season_data.get("errors")
)

print(
    "Results on this page:",
    injuries_season_data.get("results")
)

print(
    "Paging:",
    injuries_season_data.get("paging")
)


if injuries_season_data.get("response"):

    print()
    print("FIRST 3 INJURY RECORDS:")

    print(
        json.dumps(
            injuries_season_data["response"][:3],
            indent=2
        )
    )


# ---------------------------------------------------------
# 2. SEASON-LEVEL PLAYER STATISTICS
# ---------------------------------------------------------

players_season_response = requests.get(
    f"{BASE_URL}/players",
    headers=HEADERS,
    params={
        "league": PREMIER_LEAGUE_ID,
        "season": TEST_SEASON,
        "page": 1
    },
    timeout=30
)


players_season_data = (
    players_season_response.json()
)


print()
print("=" * 70)
print("SEASON PLAYER STATISTICS")
print("=" * 70)

print(
    "Errors:",
    players_season_data.get("errors")
)

print(
    "Results on this page:",
    players_season_data.get("results")
)

print(
    "Paging:",
    players_season_data.get("paging")
)


if players_season_data.get("response"):

    print()
    print("FIRST PLAYER RECORD:")

    print(
        json.dumps(
            players_season_data["response"][0],
            indent=2
        )[:8000]
    )


# ---------------------------------------------------------
# 3. QUOTA CHECK
# ---------------------------------------------------------

print()
print("=" * 70)
print("API QUOTA")
print("=" * 70)

print(
    "Requests remaining:",
    players_season_response.headers.get(
        "x-ratelimit-requests-remaining"
    )
)

print(
    "Daily request limit:",
    players_season_response.headers.get(
        "x-ratelimit-requests-limit"
    )
)

In [ ]:
# ---------------------------------------------------------
# API-FOOTBALL - STEP 4
# COLLECT PREMIER LEAGUE INJURIES
# 2022/23 THROUGH 2024/25
# ---------------------------------------------------------

import requests
import json

from pathlib import Path


# ---------------------------------------------------------
# SETTINGS
# ---------------------------------------------------------

INJURY_SEASONS = [
    2022,
    2023,
    2024
]


RAW_API_FOOTBALL_DIR = Path(
    "../data/raw/api_football"
)


INJURY_DIR = (
    RAW_API_FOOTBALL_DIR
    /
    "injuries"
)


INJURY_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ---------------------------------------------------------
# COLLECT EACH SEASON
# ---------------------------------------------------------

for season in INJURY_SEASONS:

    print()
    print("=" * 70)
    print(
        "COLLECTING SEASON:",
        season
    )
    print("=" * 70)


    response = requests.get(
        f"{BASE_URL}/injuries",
        headers=HEADERS,
        params={
            "league": PREMIER_LEAGUE_ID,
            "season": season
        },
        timeout=60
    )


    data = response.json()


    print(
        "HTTP status:",
        response.status_code
    )


    print(
        "Errors:",
        data.get("errors")
    )


    print(
        "Results:",
        data.get("results")
    )


    print(
        "Paging:",
        data.get("paging")
    )


    # -----------------------------------------------------
    # SAVE SUCCESSFUL RESPONSE
    # -----------------------------------------------------

    if (
        response.status_code == 200
        and
        not data.get("errors")
    ):

        output_path = (
            INJURY_DIR
            /
            f"premier_league_{season}_injuries.json"
        )


        with open(
            output_path,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                data,
                f,
                indent=2,
                ensure_ascii=False
            )


        print(
            "Saved:",
            output_path
        )


    else:

        print(
            "Season was NOT saved."
        )


    print(
        "Requests remaining:",
        response.headers.get(
            "x-ratelimit-requests-remaining"
        )
    )


# ---------------------------------------------------------
# CHECK SAVED FILES
# ---------------------------------------------------------

print()
print("=" * 70)
print("FILES SAVED")
print("=" * 70)


saved_files = sorted(
    INJURY_DIR.glob(
        "*.json"
    )
)


for file_path in saved_files:

    print(
        file_path.name
    )


print()

print(
    "Total files:",
    len(saved_files)
)

In [ ]:
# ---------------------------------------------------------
# API-FOOTBALL - STEP 5
# PROCESS INJURY DATA INTO MATCH-LEVEL FEATURES
# ---------------------------------------------------------

import json
import pandas as pd

from pathlib import Path


# ---------------------------------------------------------
# SETTINGS
# ---------------------------------------------------------

INJURY_DIR = Path(
    "../data/raw/api_football/injuries"
)


PROCESSED_DIR = Path(
    "../data/processed"
)


PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ---------------------------------------------------------
# LOAD ALL INJURY RECORDS
# ---------------------------------------------------------

injury_rows = []


for file_path in sorted(
    INJURY_DIR.glob(
        "premier_league_*_injuries.json"
    )
):

    with open(
        file_path,
        "r",
        encoding="utf-8"
    ) as f:

        data = json.load(f)


    for record in data.get(
        "response",
        []
    ):

        injury_rows.append(
            {
                "fixture_id":
                    record["fixture"]["id"],

                "fixture_date":
                    record["fixture"]["date"],

                "season":
                    record["league"]["season"],

                "team_id":
                    record["team"]["id"],

                "team":
                    record["team"]["name"],

                "player_id":
                    record["player"]["id"],

                "player":
                    record["player"]["name"],

                "status":
                    record["player"]["type"],

                "reason":
                    record["player"]["reason"]
            }
        )


injuries_df = pd.DataFrame(
    injury_rows
)


injuries_df["fixture_date"] = pd.to_datetime(
    injuries_df["fixture_date"],
    utc=True
)


# ---------------------------------------------------------
# BASIC CHECKS
# ---------------------------------------------------------

print(
    "Raw injury rows:",
    len(injuries_df)
)


print(
    "Unique fixtures:",
    injuries_df["fixture_id"].nunique()
)


print(
    "Unique players:",
    injuries_df["player_id"].nunique()
)


print()
print(
    "STATUS COUNTS:"
)

print(
    injuries_df["status"]
    .value_counts(
        dropna=False
    )
)


# ---------------------------------------------------------
# REMOVE DUPLICATE PLAYER-FIXTURE RECORDS
# ---------------------------------------------------------

injuries_clean = (
    injuries_df
    .drop_duplicates(
        subset=[
            "fixture_id",
            "team_id",
            "player_id",
            "status"
        ]
    )
    .copy()
)


print()
print(
    "Rows after duplicate removal:",
    len(injuries_clean)
)


# ---------------------------------------------------------
# CREATE INDICATORS
# ---------------------------------------------------------

injuries_clean[
    "is_missing_fixture"
] = (
    injuries_clean["status"]
    .eq(
        "Missing Fixture"
    )
    .astype(int)
)


injuries_clean[
    "is_questionable"
] = (
    injuries_clean["status"]
    .eq(
        "Questionable"
    )
    .astype(int)
)


# ---------------------------------------------------------
# AGGREGATE TO TEAM + FIXTURE LEVEL
# ---------------------------------------------------------

team_fixture_injuries = (
    injuries_clean
    .groupby(
        [
            "fixture_id",
            "fixture_date",
            "season",
            "team_id",
            "team"
        ],
        as_index=False
    )
    .agg(
        injury_records=(
            "player_id",
            "nunique"
        ),

        missing_fixture=(
            "is_missing_fixture",
            "sum"
        ),

        questionable=(
            "is_questionable",
            "sum"
        )
    )
)


# ---------------------------------------------------------
# SAVE
# ---------------------------------------------------------

output_path = (
    PROCESSED_DIR
    /
    "api_football_team_fixture_injuries.csv"
)


team_fixture_injuries.to_csv(
    output_path,
    index=False
)


# ---------------------------------------------------------
# FINAL CHECK
# ---------------------------------------------------------

print()
print(
    "Team-fixture rows:",
    len(team_fixture_injuries)
)


print(
    "Unique fixtures:",
    team_fixture_injuries[
        "fixture_id"
    ].nunique()
)


print()
print(
    "Average missing players per team-fixture:",
    round(
        team_fixture_injuries[
            "missing_fixture"
        ].mean(),
        3
    )
)


print(
    "Average questionable players per team-fixture:",
    round(
        team_fixture_injuries[
            "questionable"
        ].mean(),
        3
    )
)


print()
print(
    "Saved:",
    output_path
)


print()
print(
    team_fixture_injuries.head(
        10
    )
)

In [ ]:
# ---------------------------------------------------------
# API-FOOTBALL - STEP 6
# MATCH API-FOOTBALL FIXTURES TO OUR EXISTING PL DATA
# ---------------------------------------------------------

import pandas as pd
import json
import re
import unicodedata

from pathlib import Path


# ---------------------------------------------------------
# LOAD OUR EXISTING PREMIER LEAGUE MATCH DATA
# ---------------------------------------------------------

model_matches = pd.read_csv(
    "../data/processed/pl_match_features.csv"
)


model_matches["date"] = pd.to_datetime(
    model_matches["date"],
    utc=True
)


# We only have API-Football access for these seasons
model_matches = model_matches[
    model_matches["season"].isin(
        [
            "2022_23",
            "2023_24",
            "2024_25"
        ]
    )
].copy()


print(
    "Existing PL matches:",
    len(model_matches)
)


# ---------------------------------------------------------
# BUILD API-FOOTBALL FIXTURE TABLE
#
# We use the injury JSON because each injury record
# already contains the fixture ID, date and team.
# ---------------------------------------------------------

INJURY_DIR = Path(
    "../data/raw/api_football/injuries"
)


fixture_rows = []


for file_path in sorted(
    INJURY_DIR.glob(
        "premier_league_*_injuries.json"
    )
):

    with open(
        file_path,
        "r",
        encoding="utf-8"
    ) as f:

        data = json.load(f)


    for record in data.get(
        "response",
        []
    ):

        fixture_rows.append(
            {
                "api_fixture_id":
                    record["fixture"]["id"],

                "api_date":
                    record["fixture"]["date"],

                "api_season":
                    record["league"]["season"],

                "team":
                    record["team"]["name"]
            }
        )


api_fixture_teams = pd.DataFrame(
    fixture_rows
).drop_duplicates()


api_fixture_teams["api_date"] = pd.to_datetime(
    api_fixture_teams["api_date"],
    utc=True
)


# ---------------------------------------------------------
# CREATE ONE ROW PER API-FOOTBALL FIXTURE
#
# We don't yet know home/away from the injury records,
# so for this diagnostic step we'll first inspect the
# team-name overlap.
# ---------------------------------------------------------

def normalize_team_name(name):

    if pd.isna(name):
        return None

    name = str(name).lower().strip()

    name = unicodedata.normalize(
        "NFKD",
        name
    )

    name = "".join(
        char
        for char in name
        if not unicodedata.combining(char)
    )

    name = re.sub(
        r"[^a-z0-9]+",
        " ",
        name
    )

    name = re.sub(
        r"\s+",
        " ",
        name
    ).strip()

    return name


# ---------------------------------------------------------
# GET UNIQUE TEAM NAMES FROM BOTH SOURCES
# ---------------------------------------------------------

existing_teams = sorted(
    set(
        model_matches["home_team"].dropna()
    )
    |
    set(
        model_matches["away_team"].dropna()
    )
)


api_teams = sorted(
    api_fixture_teams["team"]
    .dropna()
    .unique()
)


# ---------------------------------------------------------
# NORMALIZED NAME TABLES
# ---------------------------------------------------------

existing_team_table = pd.DataFrame(
    {
        "existing_team":
            existing_teams
    }
)


existing_team_table[
    "normalized"
] = (
    existing_team_table[
        "existing_team"
    ]
    .apply(
        normalize_team_name
    )
)


api_team_table = pd.DataFrame(
    {
        "api_team":
            api_teams
    }
)


api_team_table[
    "normalized"
] = (
    api_team_table[
        "api_team"
    ]
    .apply(
        normalize_team_name
    )
)


# ---------------------------------------------------------
# EXACT NORMALIZED MATCH
# ---------------------------------------------------------

team_name_comparison = (
    existing_team_table
    .merge(
        api_team_table,
        on="normalized",
        how="outer",
        indicator=True
    )
)


matched_names = team_name_comparison[
    team_name_comparison["_merge"]
    ==
    "both"
]


existing_unmatched = team_name_comparison[
    team_name_comparison["_merge"]
    ==
    "left_only"
]


api_unmatched = team_name_comparison[
    team_name_comparison["_merge"]
    ==
    "right_only"
]


# ---------------------------------------------------------
# RESULTS
# ---------------------------------------------------------

print()
print("=" * 70)
print("TEAM NAME MATCHING")
print("=" * 70)


print(
    "Existing provider teams:",
    len(existing_teams)
)


print(
    "API-Football teams:",
    len(api_teams)
)


print(
    "Exact normalized matches:",
    len(matched_names)
)


print()
print("=" * 70)
print("UNMATCHED EXISTING TEAM NAMES")
print("=" * 70)


print(
    existing_unmatched[
        [
            "existing_team",
            "normalized"
        ]
    ].to_string(
        index=False
    )
)


print()
print("=" * 70)
print("UNMATCHED API-FOOTBALL TEAM NAMES")
print("=" * 70)


print(
    api_unmatched[
        [
            "api_team",
            "normalized"
        ]
    ].to_string(
        index=False
    )
)

In [ ]:
# ---------------------------------------------------------
# API-FOOTBALL - STEP 7
# COLLECT FIXTURE IDENTITIES + APPLY TEAM NAME MAPPING
# ---------------------------------------------------------

import requests
import json
import pandas as pd

from pathlib import Path


# ---------------------------------------------------------
# TEAM NAME MAPPING
#
# API-Football name -> our existing model name
# ---------------------------------------------------------

TEAM_NAME_MAP = {

    "Brighton":
        "Brighton & Hove Albion",

    "Ipswich":
        "Ipswich Town",

    "Leeds":
        "Leeds United",

    "Leicester":
        "Leicester City",

    "Luton":
        "Luton Town",

    "Newcastle":
        "Newcastle United",

    "Sheffield Utd":
        "Sheffield United",

    "Tottenham":
        "Tottenham Hotspur",

    "West Ham":
        "West Ham United",

    "Wolves":
        "Wolverhampton"
}


def map_api_team_name(name):

    return TEAM_NAME_MAP.get(
        name,
        name
    )


# ---------------------------------------------------------
# SETTINGS
# ---------------------------------------------------------

FIXTURE_SEASONS = [
    2022,
    2023,
    2024
]


FIXTURE_DIR = Path(
    "../data/raw/api_football/fixtures"
)


FIXTURE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


all_fixture_rows = []


# ---------------------------------------------------------
# COLLECT FIXTURES
# ---------------------------------------------------------

for season in FIXTURE_SEASONS:

    print()
    print("=" * 70)
    print(
        "COLLECTING FIXTURES:",
        season
    )
    print("=" * 70)


    response = requests.get(
        f"{BASE_URL}/fixtures",
        headers=HEADERS,
        params={
            "league":
                PREMIER_LEAGUE_ID,

            "season":
                season
        },
        timeout=60
    )


    data = response.json()


    print(
        "HTTP status:",
        response.status_code
    )


    print(
        "Errors:",
        data.get("errors")
    )


    print(
        "Results:",
        data.get("results")
    )


    # -----------------------------------------------------
    # SAVE RAW RESPONSE
    # -----------------------------------------------------

    if (
        response.status_code == 200
        and
        not data.get("errors")
    ):

        output_path = (
            FIXTURE_DIR
            /
            f"premier_league_{season}_fixtures.json"
        )


        with open(
            output_path,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                data,
                f,
                indent=2,
                ensure_ascii=False
            )


        print(
            "Saved:",
            output_path
        )


        # -------------------------------------------------
        # CREATE CLEAN FIXTURE ROWS
        # -------------------------------------------------

        for fixture in data.get(
            "response",
            []
        ):

            home_api = (
                fixture[
                    "teams"
                ][
                    "home"
                ][
                    "name"
                ]
            )


            away_api = (
                fixture[
                    "teams"
                ][
                    "away"
                ][
                    "name"
                ]
            )


            all_fixture_rows.append(
                {
                    "api_fixture_id":
                        fixture[
                            "fixture"
                        ][
                            "id"
                        ],

                    "date":
                        fixture[
                            "fixture"
                        ][
                            "date"
                        ],

                    "api_season":
                        season,

                    "home_team_api":
                        home_api,

                    "away_team_api":
                        away_api,

                    "home_team":
                        map_api_team_name(
                            home_api
                        ),

                    "away_team":
                        map_api_team_name(
                            away_api
                        )
                }
            )


    print(
        "Requests remaining:",
        response.headers.get(
            "x-ratelimit-requests-remaining"
        )
    )


# ---------------------------------------------------------
# CREATE FIXTURE DATAFRAME
# ---------------------------------------------------------

api_fixtures_df = pd.DataFrame(
    all_fixture_rows
)


api_fixtures_df[
    "date"
] = pd.to_datetime(
    api_fixtures_df["date"],
    utc=True
)


# ---------------------------------------------------------
# CHECK EACH SEASON
# ---------------------------------------------------------

print()
print("=" * 70)
print("FIXTURES BY SEASON")
print("=" * 70)


print(
    api_fixtures_df[
        "api_season"
    ]
    .value_counts()
    .sort_index()
)


print()
print(
    "Total API-Football fixtures:",
    len(api_fixtures_df)
)


print(
    "Duplicate fixture IDs:",
    api_fixtures_df[
        "api_fixture_id"
    ].duplicated().sum()
)


# ---------------------------------------------------------
# CHECK TEAM NAMES AFTER MAPPING
# ---------------------------------------------------------

api_mapped_teams = set(
    api_fixtures_df[
        "home_team"
    ]
).union(
    set(
        api_fixtures_df[
            "away_team"
        ]
    )
)


existing_teams_set = set(
    model_matches[
        "home_team"
    ]
).union(
    set(
        model_matches[
            "away_team"
        ]
    )
)


print()
print("=" * 70)
print("TEAM MAPPING CHECK")
print("=" * 70)


print(
    "Teams after mapping:",
    len(api_mapped_teams)
)


print(
    "Existing model teams:",
    len(existing_teams_set)
)


print(
    "API teams not in model:",
    sorted(
        api_mapped_teams
        -
        existing_teams_set
    )
)


print(
    "Model teams not in API:",
    sorted(
        existing_teams_set
        -
        api_mapped_teams
    )
)


# ---------------------------------------------------------
# SAVE CLEAN FIXTURE TABLE
# ---------------------------------------------------------

fixture_output_path = (
    "../data/processed/"
    "api_football_pl_fixtures.csv"
)


api_fixtures_df.to_csv(
    fixture_output_path,
    index=False
)


print()
print(
    "Saved clean fixture table:",
    fixture_output_path
)

In [ ]:
# ---------------------------------------------------------
# API-FOOTBALL - STEP 8
# BUILD MATCH-LEVEL INJURY FEATURES
# ---------------------------------------------------------

import pandas as pd


# ---------------------------------------------------------
# LOAD CLEAN TABLES
# ---------------------------------------------------------

api_fixtures_df = pd.read_csv(
    "../data/processed/api_football_pl_fixtures.csv"
)

team_fixture_injuries = pd.read_csv(
    "../data/processed/api_football_team_fixture_injuries.csv"
)


# ---------------------------------------------------------
# PREPARE HOME-SIDE INJURY TABLE
# ---------------------------------------------------------

home_injuries = (
    team_fixture_injuries
    .rename(
        columns={
            "team_id": "home_team_id",
            "team": "home_team_api",
            "injury_records": "injury_records_home",
            "missing_fixture": "missing_fixture_home",
            "questionable": "questionable_home"
        }
    )
    [
        [
            "fixture_id",
            "home_team_id",
            "home_team_api",
            "injury_records_home",
            "missing_fixture_home",
            "questionable_home"
        ]
    ]
)


# ---------------------------------------------------------
# PREPARE AWAY-SIDE INJURY TABLE
# ---------------------------------------------------------

away_injuries = (
    team_fixture_injuries
    .rename(
        columns={
            "team_id": "away_team_id",
            "team": "away_team_api",
            "injury_records": "injury_records_away",
            "missing_fixture": "missing_fixture_away",
            "questionable": "questionable_away"
        }
    )
    [
        [
            "fixture_id",
            "away_team_id",
            "away_team_api",
            "injury_records_away",
            "missing_fixture_away",
            "questionable_away"
        ]
    ]
)


# ---------------------------------------------------------
# MERGE HOME INJURIES
#
# Match on fixture ID + actual home team name.
# This prevents accidentally assigning the away team's
# injury list to the home side.
# ---------------------------------------------------------

match_injuries = api_fixtures_df.merge(
    home_injuries,
    left_on=[
        "api_fixture_id",
        "home_team_api"
    ],
    right_on=[
        "fixture_id",
        "home_team_api"
    ],
    how="left"
)


match_injuries = match_injuries.drop(
    columns=[
        "fixture_id"
    ]
)


# ---------------------------------------------------------
# MERGE AWAY INJURIES
# ---------------------------------------------------------

match_injuries = match_injuries.merge(
    away_injuries,
    left_on=[
        "api_fixture_id",
        "away_team_api"
    ],
    right_on=[
        "fixture_id",
        "away_team_api"
    ],
    how="left"
)


match_injuries = match_injuries.drop(
    columns=[
        "fixture_id"
    ]
)


# ---------------------------------------------------------
# FIX FIXTURES WITH NO LISTED INJURIES
#
# No injury row for a team means no listed injury/questionable
# players for that fixture, so fill those values with 0.
# ---------------------------------------------------------

injury_feature_columns = [
    "injury_records_home",
    "missing_fixture_home",
    "questionable_home",
    "injury_records_away",
    "missing_fixture_away",
    "questionable_away"
]


match_injuries[
    injury_feature_columns
] = (
    match_injuries[
        injury_feature_columns
    ]
    .fillna(0)
    .astype(int)
)


# ---------------------------------------------------------
# CREATE DIFFERENCE FEATURES
#
# Positive = home team has MORE availability problems
# than away team.
# ---------------------------------------------------------

match_injuries[
    "missing_fixture_difference"
] = (
    match_injuries[
        "missing_fixture_home"
    ]
    -
    match_injuries[
        "missing_fixture_away"
    ]
)


match_injuries[
    "questionable_difference"
] = (
    match_injuries[
        "questionable_home"
    ]
    -
    match_injuries[
        "questionable_away"
    ]
)


match_injuries[
    "injury_records_difference"
] = (
    match_injuries[
        "injury_records_home"
    ]
    -
    match_injuries[
        "injury_records_away"
    ]
)


# ---------------------------------------------------------
# FINAL MATCH-LEVEL TABLE
# ---------------------------------------------------------

injury_match_features = match_injuries[
    [
        "api_fixture_id",
        "date",
        "api_season",
        "home_team",
        "away_team",

        "injury_records_home",
        "missing_fixture_home",
        "questionable_home",

        "injury_records_away",
        "missing_fixture_away",
        "questionable_away",

        "injury_records_difference",
        "missing_fixture_difference",
        "questionable_difference"
    ]
].copy()


# ---------------------------------------------------------
# VALIDATION
# ---------------------------------------------------------

print(
    "Match-level rows:",
    len(injury_match_features)
)


print(
    "Unique fixture IDs:",
    injury_match_features[
        "api_fixture_id"
    ].nunique()
)


print(
    "Duplicate fixture IDs:",
    injury_match_features[
        "api_fixture_id"
    ].duplicated().sum()
)


print()

print(
    "Fixtures with zero home injury records:",
    (
        injury_match_features[
            "injury_records_home"
        ]
        ==
        0
    ).sum()
)


print(
    "Fixtures with zero away injury records:",
    (
        injury_match_features[
            "injury_records_away"
        ]
        ==
        0
    ).sum()
)


print()

print(
    "Average missing - home:",
    round(
        injury_match_features[
            "missing_fixture_home"
        ].mean(),
        3
    )
)


print(
    "Average missing - away:",
    round(
        injury_match_features[
            "missing_fixture_away"
        ].mean(),
        3
    )
)


print()

print(
    injury_match_features.head(
        10
    ).to_string(
        index=False
    )
)


# ---------------------------------------------------------
# SAVE
# ---------------------------------------------------------

output_path = (
    "../data/processed/"
    "api_football_match_injury_features.csv"
)


injury_match_features.to_csv(
    output_path,
    index=False
)


print()
print(
    "Saved:",
    output_path
)

In [ ]:
# ---------------------------------------------------------
# API-FOOTBALL - STEP 9
# MERGE INJURY FEATURES WITH EXISTING MODEL DATA
# ---------------------------------------------------------

import pandas as pd


# ---------------------------------------------------------
# LOAD DATA
# ---------------------------------------------------------

model_df = pd.read_csv(
    "../data/processed/pl_match_features.csv"
)


injury_features = pd.read_csv(
    "../data/processed/api_football_match_injury_features.csv"
)


# ---------------------------------------------------------
# CONVERT DATES
# ---------------------------------------------------------

model_df["date"] = pd.to_datetime(
    model_df["date"],
    utc=True
)


injury_features["date"] = pd.to_datetime(
    injury_features["date"],
    utc=True
)


# ---------------------------------------------------------
# CREATE CALENDAR-DATE MATCHING KEY
#
# We intentionally ignore exact kickoff time.
# ---------------------------------------------------------

model_df["match_date"] = (
    model_df["date"]
    .dt.date
)


injury_features["match_date"] = (
    injury_features["date"]
    .dt.date
)


# ---------------------------------------------------------
# ONLY THE THREE SEASONS AVAILABLE FROM API-FOOTBALL
# ---------------------------------------------------------

api_seasons = [
    "2022_23",
    "2023_24",
    "2024_25"
]


model_api_period = (
    model_df[
        model_df["season"].isin(
            api_seasons
        )
    ]
    .copy()
)


print(
    "Existing model matches:",
    len(model_api_period)
)


print(
    "API-Football injury matches:",
    len(injury_features)
)


# ---------------------------------------------------------
# SELECT INJURY COLUMNS TO ADD
# ---------------------------------------------------------

injury_columns = [
    "api_fixture_id",
    "match_date",
    "home_team",
    "away_team",

    "injury_records_home",
    "missing_fixture_home",
    "questionable_home",

    "injury_records_away",
    "missing_fixture_away",
    "questionable_away",

    "injury_records_difference",
    "missing_fixture_difference",
    "questionable_difference"
]


injury_merge_table = (
    injury_features[
        injury_columns
    ]
    .copy()
)


# ---------------------------------------------------------
# CHECK MERGE KEY UNIQUENESS BEFORE MERGING
# ---------------------------------------------------------

model_key_duplicates = (
    model_api_period
    .duplicated(
        subset=[
            "match_date",
            "home_team",
            "away_team"
        ]
    )
    .sum()
)


api_key_duplicates = (
    injury_merge_table
    .duplicated(
        subset=[
            "match_date",
            "home_team",
            "away_team"
        ]
    )
    .sum()
)


print()
print(
    "Model merge-key duplicates:",
    model_key_duplicates
)


print(
    "API merge-key duplicates:",
    api_key_duplicates
)


# ---------------------------------------------------------
# MERGE
# ---------------------------------------------------------

model_with_injuries = (
    model_api_period
    .merge(
        injury_merge_table,
        on=[
            "match_date",
            "home_team",
            "away_team"
        ],
        how="left",
        validate="one_to_one",
        indicator=True
    )
)


# ---------------------------------------------------------
# VALIDATE MATCHING
# ---------------------------------------------------------

print()
print("=" * 70)
print("MERGE RESULTS")
print("=" * 70)


print(
    model_with_injuries[
        "_merge"
    ]
    .value_counts()
)


matched = (
    model_with_injuries[
        "_merge"
    ]
    ==
    "both"
).sum()


unmatched = (
    model_with_injuries[
        "_merge"
    ]
    !=
    "both"
).sum()


print()
print(
    "Matched:",
    matched,
    "/",
    len(model_with_injuries)
)


print(
    "Unmatched:",
    unmatched
)


# ---------------------------------------------------------
# SHOW UNMATCHED MATCHES IF ANY
# ---------------------------------------------------------

if unmatched > 0:

    print()
    print("=" * 70)
    print("UNMATCHED MATCHES")
    print("=" * 70)

    print(
        model_with_injuries.loc[
            model_with_injuries[
                "_merge"
            ]
            !=
            "both",
            [
                "date",
                "season",
                "home_team",
                "away_team"
            ]
        ]
        .to_string(
            index=False
        )
    )


# ---------------------------------------------------------
# CHECK INJURY FEATURE MISSINGNESS
# ---------------------------------------------------------

new_injury_features = [
    "injury_records_home",
    "missing_fixture_home",
    "questionable_home",

    "injury_records_away",
    "missing_fixture_away",
    "questionable_away",

    "injury_records_difference",
    "missing_fixture_difference",
    "questionable_difference"
]


print()
print("=" * 70)
print("INJURY FEATURE MISSING VALUES")
print("=" * 70)


print(
    model_with_injuries[
        new_injury_features
    ]
    .isna()
    .sum()
)


# ---------------------------------------------------------
# REMOVE MERGE INDICATOR
# ---------------------------------------------------------

model_with_injuries = (
    model_with_injuries
    .drop(
        columns=[
            "_merge"
        ]
    )
)


# ---------------------------------------------------------
# SAVE EXPERIMENTAL DATASET
#
# Keep this separate from the locked V1 dataset.
# ---------------------------------------------------------

output_path = (
    "../data/processed/"
    "pl_match_features_with_injuries_experimental.csv"
)


model_with_injuries.to_csv(
    output_path,
    index=False
)


print()
print(
    "Experimental dataset shape:",
    model_with_injuries.shape
)


print(
    "Saved:",
    output_path
)

In [ ]:
# ---------------------------------------------------------
# API-FOOTBALL - STEP 11
# TEST PLAYER-STATS COLLECTION OPTIONS
# ---------------------------------------------------------

import requests
import pandas as pd


# ---------------------------------------------------------
# LOAD OUR API FIXTURE TABLE
# ---------------------------------------------------------

api_fixtures_df = pd.read_csv(
    "../data/processed/api_football_pl_fixtures.csv"
)


api_fixtures_df["date"] = pd.to_datetime(
    api_fixtures_df["date"],
    utc=True
)


# ---------------------------------------------------------
# PICK ONE HISTORICAL FIXTURE
# ---------------------------------------------------------

sample_fixture = (
    api_fixtures_df[
        api_fixtures_df["api_season"]
        ==
        2024
    ]
    .iloc[0]
)


sample_fixture_id = int(
    sample_fixture["api_fixture_id"]
)


sample_date = (
    sample_fixture["date"]
    .strftime("%Y-%m-%d")
)


print("=" * 70)
print("SAMPLE FIXTURE")
print("=" * 70)

print(
    sample_fixture[
        [
            "api_fixture_id",
            "date",
            "home_team",
            "away_team"
        ]
    ]
)

print()

print(
    "Fixture ID:",
    sample_fixture_id
)

print(
    "Fixture date:",
    sample_date
)


# =========================================================
# TEST 1
# STANDARD SINGLE-FIXTURE REQUEST
#
# We know this should work.
# This gives us a baseline for comparison.
# =========================================================

print()
print("=" * 70)
print("TEST 1 — SINGLE FIXTURE")
print("=" * 70)


response_1 = requests.get(

    f"{BASE_URL}/fixtures/players",

    headers=HEADERS,

    params={
        "fixture":
            sample_fixture_id
    },

    timeout=60
)


data_1 = response_1.json()


print(
    "HTTP status:",
    response_1.status_code
)


print(
    "Errors:",
    data_1.get("errors")
)


print(
    "Results:",
    data_1.get("results")
)


print(
    "Requests remaining:",
    response_1.headers.get(
        "x-ratelimit-requests-remaining"
    )
)


# =========================================================
# TEST 2
# TRY LEAGUE + SEASON
#
# If supported, this would be extremely useful because
# we could potentially retrieve many fixtures at once.
# =========================================================

print()
print("=" * 70)
print("TEST 2 — LEAGUE + SEASON")
print("=" * 70)


response_2 = requests.get(

    f"{BASE_URL}/fixtures/players",

    headers=HEADERS,

    params={
        "league":
            PREMIER_LEAGUE_ID,

        "season":
            2024
    },

    timeout=60
)


data_2 = response_2.json()


print(
    "HTTP status:",
    response_2.status_code
)


print(
    "Errors:",
    data_2.get("errors")
)


print(
    "Results:",
    data_2.get("results")
)


print(
    "Paging:",
    data_2.get("paging")
)


print(
    "Requests remaining:",
    response_2.headers.get(
        "x-ratelimit-requests-remaining"
    )
)


# =========================================================
# TEST 3
# TRY LEAGUE + DATE
#
# If this works, one request could potentially retrieve
# all player performances from a Premier League matchday.
# =========================================================

print()
print("=" * 70)
print("TEST 3 — LEAGUE + DATE")
print("=" * 70)


response_3 = requests.get(

    f"{BASE_URL}/fixtures/players",

    headers=HEADERS,

    params={
        "league":
            PREMIER_LEAGUE_ID,

        "date":
            sample_date
    },

    timeout=60
)


data_3 = response_3.json()


print(
    "HTTP status:",
    response_3.status_code
)


print(
    "Errors:",
    data_3.get("errors")
)


print(
    "Results:",
    data_3.get("results")
)


print(
    "Paging:",
    data_3.get("paging")
)


print(
    "Requests remaining:",
    response_3.headers.get(
        "x-ratelimit-requests-remaining"
    )
)

In [ ]:
# ---------------------------------------------------------
# API-FOOTBALL - STEP 12
# DOES INJURY DISADVANTAGE RELATE TO MATCH RESULTS?
# ---------------------------------------------------------

import pandas as pd
import numpy as np


# ---------------------------------------------------------
# LOAD EXPERIMENTAL DATASET
# ---------------------------------------------------------

df = pd.read_csv(
    "../data/processed/"
    "pl_match_features_with_injuries_experimental.csv"
)


# ---------------------------------------------------------
# HOME POINTS
# ---------------------------------------------------------

df["home_points"] = np.select(

    [
        df["target"] == "H",
        df["target"] == "D",
        df["target"] == "A"
    ],

    [
        3,
        1,
        0
    ],

    default=np.nan
)


# ---------------------------------------------------------
# HOME WIN INDICATOR
# ---------------------------------------------------------

df["home_win"] = (
    df["target"]
    ==
    "H"
).astype(int)


# ---------------------------------------------------------
# CREATE INJURY-DISADVANTAGE GROUPS
#
# missing_fixture_difference:
#
# positive = HOME has more missing players
# negative = AWAY has more missing players
# ---------------------------------------------------------

df["injury_disadvantage_group"] = pd.cut(

    df["missing_fixture_difference"],

    bins=[
        -np.inf,
        -4,
        -2,
        1,
        3,
        np.inf
    ],

    labels=[
        "Home 5+ healthier",
        "Home 3-4 healthier",
        "Roughly even",
        "Home 2-3 more missing",
        "Home 4+ more missing"
    ]
)


# ---------------------------------------------------------
# GROUP RESULTS
# ---------------------------------------------------------

injury_result_summary = (

    df.groupby(
        "injury_disadvantage_group",
        observed=True
    )

    .agg(

        matches=(
            "match_id",
            "count"
        ),

        avg_home_missing=(
            "missing_fixture_home",
            "mean"
        ),

        avg_away_missing=(
            "missing_fixture_away",
            "mean"
        ),

        home_win_rate=(
            "home_win",
            "mean"
        ),

        avg_home_points=(
            "home_points",
            "mean"
        )
    )

    .reset_index()
)


# ---------------------------------------------------------
# ROUND FOR DISPLAY
# ---------------------------------------------------------

injury_result_summary[
    "avg_home_missing"
] = injury_result_summary[
    "avg_home_missing"
].round(2)


injury_result_summary[
    "avg_away_missing"
] = injury_result_summary[
    "avg_away_missing"
].round(2)


injury_result_summary[
    "home_win_rate"
] = injury_result_summary[
    "home_win_rate"
].round(3)


injury_result_summary[
    "avg_home_points"
] = injury_result_summary[
    "avg_home_points"
].round(3)


# ---------------------------------------------------------
# PRINT RESULTS
# ---------------------------------------------------------

print("=" * 75)
print("INJURY DISADVANTAGE VS ACTUAL RESULTS")
print("=" * 75)


print(
    injury_result_summary.to_string(
        index=False
    )
)


# ---------------------------------------------------------
# SIMPLE CORRELATIONS
# ---------------------------------------------------------

print()
print("=" * 75)
print("SIMPLE CORRELATIONS")
print("=" * 75)


print(
    "Missing-player difference vs home points:",
    round(
        df[
            "missing_fixture_difference"
        ].corr(
            df[
                "home_points"
            ]
        ),
        4
    )
)


print(
    "Missing-player difference vs home win:",
    round(
        df[
            "missing_fixture_difference"
        ].corr(
            df[
                "home_win"
            ]
        ),
        4
    )
)


# ---------------------------------------------------------
# SEASON-BY-SEASON CHECK
# ---------------------------------------------------------

season_summary = (

    df.groupby(
        "season"
    )

    .agg(

        matches=(
            "match_id",
            "count"
        ),

        avg_home_missing=(
            "missing_fixture_home",
            "mean"
        ),

        avg_away_missing=(
            "missing_fixture_away",
            "mean"
        ),

        avg_missing_difference=(
            "missing_fixture_difference",
            "mean"
        ),

        home_win_rate=(
            "home_win",
            "mean"
        )
    )

    .reset_index()
)


print()
print("=" * 75)
print("SEASON SUMMARY")
print("=" * 75)


print(
    season_summary.round(
        3
    ).to_string(
        index=False
    )
)

In [ ]:
# ---------------------------------------------------------
# API-FOOTBALL - STEP 13
# INSPECT V2 INJURY FEATURE IMPORTANCE
# ---------------------------------------------------------

import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer


# ---------------------------------------------------------
# LOAD DATA
# ---------------------------------------------------------

df = pd.read_csv(
    "../data/processed/"
    "pl_match_features_with_injuries_experimental.csv"
)


df["date"] = pd.to_datetime(
    df["date"],
    utc=True
)


df = (
    df
    .sort_values(
        [
            "date",
            "match_id"
        ]
    )
    .reset_index(
        drop=True
    )
)


# ---------------------------------------------------------
# LOCKED V1 FEATURES
# ---------------------------------------------------------

baseline_features = [

    "points_last5_all_home",
    "goal_difference_last5_all_home",
    "win_rate_last5_all_home",
    "shots_on_target_difference_last5_all_home",

    "points_last5_all_away",
    "goal_difference_last5_all_away",
    "win_rate_last5_all_away",
    "shots_on_target_difference_last5_all_away",

    "points_last5_pl_home",
    "goal_difference_last5_pl_home",
    "win_rate_last5_pl_home",
    "shots_on_target_difference_last5_pl_home",
    "points_last10_pl_home",
    "goal_difference_last10_pl_home",
    "win_rate_last10_pl_home",

    "points_last5_pl_away",
    "goal_difference_last5_pl_away",
    "win_rate_last5_pl_away",
    "shots_on_target_difference_last5_pl_away",
    "points_last10_pl_away",
    "goal_difference_last10_pl_away",
    "win_rate_last10_pl_away",

    "days_rest_home",
    "days_rest_away",

    "matches_last_7_days_home",
    "matches_last_7_days_away",

    "matches_last_14_days_home",
    "matches_last_14_days_away",

    "team_elo_home",
    "team_elo_away",
    "team_elo_difference_home"
]


# ---------------------------------------------------------
# INJURY FEATURES
# ---------------------------------------------------------

injury_features = [

    "missing_fixture_home",
    "questionable_home",

    "missing_fixture_away",
    "questionable_away",

    "missing_fixture_difference",
    "questionable_difference"
]


v2_features = (
    baseline_features
    +
    injury_features
)


# ---------------------------------------------------------
# TRAIN ON 2022/23 + 2023/24
#
# We stop before 2024/25 so the setup remains consistent
# with our final walk-forward test.
# ---------------------------------------------------------

train_df = (
    df[
        df["season"].isin(
            [
                "2022_23",
                "2023_24"
            ]
        )
    ]
    .copy()
)


print(
    "Training matches:",
    len(train_df)
)


# ---------------------------------------------------------
# IMPUTE
# ---------------------------------------------------------

imputer = SimpleImputer(
    strategy="median",
    add_indicator=True
)


X_train = (
    imputer.fit_transform(
        train_df[
            v2_features
        ]
    )
)


y_train = (
    train_df["target"]
)


# ---------------------------------------------------------
# TRAIN SAME LOCKED RF
# ---------------------------------------------------------

model = RandomForestClassifier(

    n_estimators=500,

    max_depth=6,

    min_samples_leaf=10,

    max_features="sqrt",

    criterion="gini",

    class_weight=None,

    random_state=42,

    n_jobs=-1
)


model.fit(
    X_train,
    y_train
)


# ---------------------------------------------------------
# GET ACTUAL FEATURE NAMES AFTER IMPUTATION
#
# This also handles any missing-value indicator columns
# automatically created by SimpleImputer.
# ---------------------------------------------------------

transformed_feature_names = (
    imputer.get_feature_names_out(
        v2_features
    )
)


importance_df = pd.DataFrame(
    {
        "feature":
            transformed_feature_names,

        "importance":
            model.feature_importances_
    }
)


importance_df = (
    importance_df
    .sort_values(
        "importance",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


importance_df[
    "rank"
] = (
    importance_df.index
    +
    1
)


# ---------------------------------------------------------
# TOP 20 FEATURES OVERALL
# ---------------------------------------------------------

print()
print("=" * 80)
print("TOP 20 V2 FEATURES")
print("=" * 80)


print(
    importance_df[
        [
            "rank",
            "feature",
            "importance"
        ]
    ]
    .head(20)
    .to_string(
        index=False
    )
)


# ---------------------------------------------------------
# INJURY FEATURES ONLY
# ---------------------------------------------------------

injury_importance = (
    importance_df[
        importance_df[
            "feature"
        ].isin(
            injury_features
        )
    ]
    .copy()
)


print()
print("=" * 80)
print("INJURY FEATURE IMPORTANCE")
print("=" * 80)


print(
    injury_importance[
        [
            "rank",
            "feature",
            "importance"
        ]
    ]
    .to_string(
        index=False
    )
)


# ---------------------------------------------------------
# TOTAL IMPORTANCE GIVEN TO INJURY FEATURES
# ---------------------------------------------------------

total_injury_importance = (
    injury_importance[
        "importance"
    ].sum()
)


print()

print(
    "Total importance assigned to injury features:",
    round(
        total_injury_importance,
        4
    )
)


print(
    "Percent of total RF importance:",
    round(
        total_injury_importance
        *
        100,
        2
    ),
    "%"
)

In [ ]:
# ---------------------------------------------------------
# API-FOOTBALL - STEP 14
# ANALYZE WHICH PLAYERS DRIVE THE INJURY DATA
# ---------------------------------------------------------

import json
import pandas as pd

from pathlib import Path


# ---------------------------------------------------------
# LOAD RAW INJURY FILES
# ---------------------------------------------------------

injury_dir = Path(
    "../data/raw/api_football/injuries"
)


seasons = [
    2022,
    2023,
    2024
]


rows = []


for season in seasons:

    path = (
        injury_dir
        /
        f"premier_league_{season}_injuries.json"
    )


    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        data = json.load(f)


    for record in data.get(
        "response",
        []
    ):

        player = record.get(
            "player",
            {}
        )

        team = record.get(
            "team",
            {}
        )

        fixture = record.get(
            "fixture",
            {}
        )


        rows.append(
            {
                "season":
                    season,

                "fixture_id":
                    fixture.get(
                        "id"
                    ),

                "fixture_date":
                    fixture.get(
                        "date"
                    ),

                "team_id":
                    team.get(
                        "id"
                    ),

                "team":
                    team.get(
                        "name"
                    ),

                "player_id":
                    player.get(
                        "id"
                    ),

                "player":
                    player.get(
                        "name"
                    ),

                "status":
                    player.get(
                        "type"
                    ),

                "reason":
                    player.get(
                        "reason"
                    )
            }
        )


injury_players = pd.DataFrame(
    rows
)


# ---------------------------------------------------------
# ONLY CONFIRMED MISSING PLAYERS
#
# We focus on Missing Fixture because it was more useful
# than Questionable in our RF importance analysis.
# ---------------------------------------------------------

missing_players = (
    injury_players[
        injury_players["status"]
        ==
        "Missing Fixture"
    ]
    .copy()
)


# ---------------------------------------------------------
# PLAYER-LEVEL SUMMARY
# ---------------------------------------------------------

player_summary = (

    missing_players

    .groupby(
        [
            "player_id",
            "player"
        ],
        as_index=False
    )

    .agg(

        missing_fixture_records=(
            "fixture_id",
            "nunique"
        ),

        teams=(
            "team_id",
            "nunique"
        ),

        seasons=(
            "season",
            "nunique"
        )
    )

    .sort_values(
        "missing_fixture_records",
        ascending=False
    )

    .reset_index(
        drop=True
    )
)


# ---------------------------------------------------------
# GENERAL COUNTS
# ---------------------------------------------------------

print("=" * 80)
print("CONFIRMED MISSING-PLAYER SUMMARY")
print("=" * 80)


print(
    "Missing-fixture records:",
    len(
        missing_players
    )
)


print(
    "Unique missing players:",
    missing_players[
        "player_id"
    ].nunique()
)


print(
    "Unique teams:",
    missing_players[
        "team_id"
    ].nunique()
)


# ---------------------------------------------------------
# MOST FREQUENTLY MISSING PLAYERS
# ---------------------------------------------------------

print()
print("=" * 80)
print("TOP 30 MOST FREQUENTLY MISSING PLAYERS")
print("=" * 80)


print(
    player_summary
    .head(30)
    .to_string(
        index=False
    )
)


# ---------------------------------------------------------
# DISTRIBUTION OF MISSING MATCHES
# ---------------------------------------------------------

print()
print("=" * 80)
print("MISSING-MATCH DISTRIBUTION")
print("=" * 80)


thresholds = [
    1,
    3,
    5,
    10,
    15,
    20
]


for threshold in thresholds:

    count = (
        player_summary[
            "missing_fixture_records"
        ]
        >=
        threshold
    ).sum()


    print(
        f"Players missing {threshold}+ fixtures:",
        count
    )


# ---------------------------------------------------------
# HOW CONCENTRATED ARE THE INJURY RECORDS?
# ---------------------------------------------------------

total_missing_records = (
    player_summary[
        "missing_fixture_records"
    ].sum()
)


print()
print("=" * 80)
print("CONCENTRATION OF MISSING-FIXTURE RECORDS")
print("=" * 80)


for top_n in [
    25,
    50,
    100,
    200,
    300
]:

    top_records = (
        player_summary
        .head(
            top_n
        )[
            "missing_fixture_records"
        ]
        .sum()
    )


    share = (
        top_records
        /
        total_missing_records
    )


    print(
        f"Top {top_n} players:",
        f"{share:.2%}"
    )


# ---------------------------------------------------------
# SAVE PLAYER SUMMARY
# ---------------------------------------------------------

output_path = (
    "../data/processed/"
    "api_football_missing_player_summary.csv"
)


player_summary.to_csv(
    output_path,
    index=False
)


print()
print(
    "Saved:",
    output_path
)

In [ ]:
# ---------------------------------------------------------
# API-FOOTBALL - STEP 15
# DOES INJURY DISADVANTAGE ADD SIGNAL BEYOND ELO?
# ---------------------------------------------------------

import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    log_loss,
    roc_auc_score
)


# ---------------------------------------------------------
# LOAD DATA
# ---------------------------------------------------------

df = pd.read_csv(
    "../data/processed/"
    "pl_match_features_with_injuries_experimental.csv"
)


df["date"] = pd.to_datetime(
    df["date"],
    utc=True
)


df = (
    df
    .sort_values(
        [
            "date",
            "match_id"
        ]
    )
    .reset_index(
        drop=True
    )
)


# ---------------------------------------------------------
# HOME-WIN TARGET
#
# 1 = home win
# 0 = draw or away win
# ---------------------------------------------------------

df["home_win_binary"] = (
    df["target"]
    ==
    "H"
).astype(int)


# ---------------------------------------------------------
# MODEL 1
# ELO ONLY
# ---------------------------------------------------------

elo_features = [
    "team_elo_difference_home"
]


# ---------------------------------------------------------
# MODEL 2
# ELO + CONFIRMED INJURY DIFFERENCE
#
# Positive injury difference means the HOME team has
# more confirmed missing players.
# ---------------------------------------------------------

elo_injury_features = [
    "team_elo_difference_home",
    "missing_fixture_difference"
]


# ---------------------------------------------------------
# WALK-FORWARD TESTS
# ---------------------------------------------------------

test_seasons = [
    "2023_24",
    "2024_25"
]


results = []


for test_season in test_seasons:

    print()
    print("=" * 75)
    print(
        "TEST SEASON:",
        test_season
    )
    print("=" * 75)


    test_start = (
        df.loc[
            df["season"]
            ==
            test_season,
            "date"
        ]
        .min()
    )


    train_df = (
        df[
            df["date"]
            <
            test_start
        ]
        .copy()
    )


    test_df = (
        df[
            df["season"]
            ==
            test_season
        ]
        .copy()
    )


    y_train = (
        train_df[
            "home_win_binary"
        ]
    )


    y_test = (
        test_df[
            "home_win_binary"
        ]
    )


    # =====================================================
    # ELO-ONLY MODEL
    # =====================================================

    elo_model = LogisticRegression(
        max_iter=1000,
        random_state=42
    )


    elo_model.fit(
        train_df[
            elo_features
        ],
        y_train
    )


    elo_prob = (
        elo_model.predict_proba(
            test_df[
                elo_features
            ]
        )[:, 1]
    )


    elo_pred = (
        elo_prob
        >=
        0.5
    ).astype(int)


    # =====================================================
    # ELO + INJURY MODEL
    # =====================================================

    injury_model = LogisticRegression(
        max_iter=1000,
        random_state=42
    )


    injury_model.fit(
        train_df[
            elo_injury_features
        ],
        y_train
    )


    injury_prob = (
        injury_model.predict_proba(
            test_df[
                elo_injury_features
            ]
        )[:, 1]
    )


    injury_pred = (
        injury_prob
        >=
        0.5
    ).astype(int)


    # =====================================================
    # METRICS
    # =====================================================

    elo_accuracy = accuracy_score(
        y_test,
        elo_pred
    )


    injury_accuracy = accuracy_score(
        y_test,
        injury_pred
    )


    elo_loss = log_loss(
        y_test,
        elo_prob
    )


    injury_loss = log_loss(
        y_test,
        injury_prob
    )


    elo_auc = roc_auc_score(
        y_test,
        elo_prob
    )


    injury_auc = roc_auc_score(
        y_test,
        injury_prob
    )


    # =====================================================
    # INJURY COEFFICIENT
    # =====================================================

    injury_coefficient = (
        injury_model.coef_[
            0
        ][
            1
        ]
    )


    print(
        "Training matches:",
        len(train_df)
    )


    print(
        "Test matches:",
        len(test_df)
    )


    print()

    print(
        "Elo-only Accuracy:",
        round(
            elo_accuracy,
            4
        )
    )


    print(
        "Elo + Injury Accuracy:",
        round(
            injury_accuracy,
            4
        )
    )


    print()

    print(
        "Elo-only Log Loss:",
        round(
            elo_loss,
            4
        )
    )


    print(
        "Elo + Injury Log Loss:",
        round(
            injury_loss,
            4
        )
    )


    print()

    print(
        "Elo-only ROC AUC:",
        round(
            elo_auc,
            4
        )
    )


    print(
        "Elo + Injury ROC AUC:",
        round(
            injury_auc,
            4
        )
    )


    print()

    print(
        "Injury-difference coefficient:",
        round(
            injury_coefficient,
            4
        )
    )


    results.append(
        {
            "season":
                test_season,

            "elo_accuracy":
                elo_accuracy,

            "elo_injury_accuracy":
                injury_accuracy,

            "elo_logloss":
                elo_loss,

            "elo_injury_logloss":
                injury_loss,

            "elo_auc":
                elo_auc,

            "elo_injury_auc":
                injury_auc,

            "injury_coefficient":
                injury_coefficient
        }
    )


# ---------------------------------------------------------
# FINAL SUMMARY
# ---------------------------------------------------------

results_df = pd.DataFrame(
    results
)


print()
print("=" * 75)
print("FINAL SUMMARY")
print("=" * 75)


print(
    results_df.to_string(
        index=False
    )
)


print()

print(
    "Average Elo-only Log Loss:",
    round(
        results_df[
            "elo_logloss"
        ].mean(),
        4
    )
)


print(
    "Average Elo + Injury Log Loss:",
    round(
        results_df[
            "elo_injury_logloss"
        ].mean(),
        4
    )
)


print()

print(
    "Average Elo-only ROC AUC:",
    round(
        results_df[
            "elo_auc"
        ].mean(),
        4
    )
)


print(
    "Average Elo + Injury ROC AUC:",
    round(
        results_df[
            "elo_injury_auc"
        ].mean(),
        4
    )
)

In [ ]:
# ---------------------------------------------------------
# API-FOOTBALL - STEP 16
# ANALYZE INJURY / ABSENCE REASONS
# ---------------------------------------------------------

import json
import pandas as pd

from pathlib import Path


# ---------------------------------------------------------
# LOAD RAW INJURY FILES
# ---------------------------------------------------------

injury_dir = Path(
    "../data/raw/api_football/injuries"
)


seasons = [
    2022,
    2023,
    2024
]


rows = []


for season in seasons:

    path = (
        injury_dir
        /
        f"premier_league_{season}_injuries.json"
    )


    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        data = json.load(f)


    for record in data.get(
        "response",
        []
    ):

        player = record.get(
            "player",
            {}
        )

        team = record.get(
            "team",
            {}
        )

        fixture = record.get(
            "fixture",
            {}
        )


        rows.append(
            {
                "season":
                    season,

                "fixture_id":
                    fixture.get(
                        "id"
                    ),

                "fixture_date":
                    fixture.get(
                        "date"
                    ),

                "team_id":
                    team.get(
                        "id"
                    ),

                "team":
                    team.get(
                        "name"
                    ),

                "player_id":
                    player.get(
                        "id"
                    ),

                "player":
                    player.get(
                        "name"
                    ),

                "status":
                    player.get(
                        "type"
                    ),

                "reason":
                    player.get(
                        "reason"
                    )
            }
        )


injury_detail_df = pd.DataFrame(
    rows
)


# ---------------------------------------------------------
# BASIC REASON COVERAGE
# ---------------------------------------------------------

print("=" * 80)
print("REASON COVERAGE")
print("=" * 80)


print(
    "Total injury records:",
    len(injury_detail_df)
)


print(
    "Records with a reason:",
    injury_detail_df[
        "reason"
    ].notna().sum()
)


print(
    "Records missing a reason:",
    injury_detail_df[
        "reason"
    ].isna().sum()
)


print(
    "Unique reasons:",
    injury_detail_df[
        "reason"
    ].nunique()
)


# ---------------------------------------------------------
# MOST COMMON REASONS
# ---------------------------------------------------------

reason_counts = (

    injury_detail_df[
        "reason"
    ]

    .fillna(
        "MISSING"
    )

    .value_counts()

    .rename_axis(
        "reason"
    )

    .reset_index(
        name="records"
    )
)


reason_counts[
    "percent"
] = (
    reason_counts[
        "records"
    ]
    /
    len(
        injury_detail_df
    )
    *
    100
)


print()
print("=" * 80)
print("TOP 40 REASONS")
print("=" * 80)


print(
    reason_counts
    .head(40)
    .round(
        {
            "percent":
                2
        }
    )
    .to_string(
        index=False
    )
)


# ---------------------------------------------------------
# CONFIRMED MISSING PLAYERS ONLY
# ---------------------------------------------------------

confirmed_missing = (
    injury_detail_df[
        injury_detail_df[
            "status"
        ]
        ==
        "Missing Fixture"
    ]
    .copy()
)


missing_reason_counts = (

    confirmed_missing[
        "reason"
    ]

    .fillna(
        "MISSING"
    )

    .value_counts()

    .rename_axis(
        "reason"
    )

    .reset_index(
        name="records"
    )
)


missing_reason_counts[
    "percent"
] = (
    missing_reason_counts[
        "records"
    ]
    /
    len(
        confirmed_missing
    )
    *
    100
)


print()
print("=" * 80)
print("TOP REASONS — CONFIRMED MISSING ONLY")
print("=" * 80)


print(
    missing_reason_counts
    .head(40)
    .round(
        {
            "percent":
                2
        }
    )
    .to_string(
        index=False
    )
)


# ---------------------------------------------------------
# STATUS BY REASON
# ---------------------------------------------------------

reason_status = pd.crosstab(

    injury_detail_df[
        "reason"
    ].fillna(
        "MISSING"
    ),

    injury_detail_df[
        "status"
    ]
)


reason_status[
    "total"
] = (
    reason_status.sum(
        axis=1
    )
)


reason_status = (
    reason_status
    .sort_values(
        "total",
        ascending=False
    )
)


print()
print("=" * 80)
print("STATUS BY TOP 30 REASONS")
print("=" * 80)


print(
    reason_status
    .head(30)
    .to_string()
)


# ---------------------------------------------------------
# REASON COVERAGE BY SEASON
# ---------------------------------------------------------

season_reason_summary = (

    injury_detail_df

    .groupby(
        "season"
    )

    .agg(

        records=(
            "fixture_id",
            "count"
        ),

        unique_reasons=(
            "reason",
            "nunique"
        ),

        reason_available=(
            "reason",
            lambda x:
                x.notna().sum()
        )
    )

    .reset_index()
)


season_reason_summary[
    "reason_coverage"
] = (
    season_reason_summary[
        "reason_available"
    ]
    /
    season_reason_summary[
        "records"
    ]
)


print()
print("=" * 80)
print("REASON COVERAGE BY SEASON")
print("=" * 80)


print(
    season_reason_summary
    .round(
        {
            "reason_coverage":
                3
        }
    )
    .to_string(
        index=False
    )
)

In [ ]:
import pandas as pd

fixtures = pd.read_csv(
    "../data/processed/api_football_pl_fixtures.csv"
)

print("Columns:")
print(fixtures.columns.tolist())

print()
print("First 5 rows:")
print(fixtures.head().to_string())

In [ ]:
# ---------------------------------------------------------
# API-FOOTBALL - STEP 17 (FINAL CORRECTED)
# BUILD MATCH-LEVEL ABSENCE TYPE FEATURES
# ---------------------------------------------------------

import json
import pandas as pd

from pathlib import Path


# ---------------------------------------------------------
# LOAD RAW INJURY DATA
# ---------------------------------------------------------

injury_dir = Path(
    "../data/raw/api_football/injuries"
)

seasons = [
    2022,
    2023,
    2024
]

rows = []


for season in seasons:

    path = (
        injury_dir
        /
        f"premier_league_{season}_injuries.json"
    )

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        data = json.load(f)

    for record in data.get(
        "response",
        []
    ):

        player = record.get(
            "player",
            {}
        )

        team = record.get(
            "team",
            {}
        )

        fixture = record.get(
            "fixture",
            {}
        )

        rows.append(
            {
                "season":
                    season,

                "fixture_id":
                    fixture.get("id"),

                "team":
                    team.get("name"),

                "player_id":
                    player.get("id"),

                "player":
                    player.get("name"),

                "status":
                    player.get("type"),

                "reason":
                    player.get("reason")
            }
        )


injury_df = pd.DataFrame(
    rows
)


# ---------------------------------------------------------
# KEEP CONFIRMED ABSENCES ONLY
# ---------------------------------------------------------

missing_df = (
    injury_df[
        injury_df["status"]
        ==
        "Missing Fixture"
    ]
    .copy()
)


# ---------------------------------------------------------
# CLASSIFY ABSENCE REASONS
# ---------------------------------------------------------

def classify_absence_reason(reason):

    reason = str(reason).lower()


    # -----------------------------------------------------
    # DISCIPLINARY
    # -----------------------------------------------------

    if any(
        word in reason
        for word in [
            "red card",
            "yellow card",
            "suspend"
        ]
    ):
        return "disciplinary"


    # -----------------------------------------------------
    # ILLNESS / FITNESS
    # -----------------------------------------------------

    if any(
        word in reason
        for word in [
            "illness",
            "health",
            "match fitness",
            "fitness"
        ]
    ):
        return "illness_fitness"


    # -----------------------------------------------------
    # SELECTION / OTHER UNAVAILABILITY
    # -----------------------------------------------------

    if any(
        word in reason
        for word in [
            "coach",
            "national selection",
            "international duty",
            "loan agreement",
            "personal",
            "inactive",
            "rest"
        ]
    ):
        return "selection_unavailable"


    # -----------------------------------------------------
    # PHYSICAL INJURY
    # -----------------------------------------------------

    if any(
        word in reason
        for word in [
            "injury",
            "knock",
            "achilles",
            "broken",
            "surgery",
            "concussion",
            "heart",
            "pelvis"
        ]
    ):
        return "physical_injury"


    # -----------------------------------------------------
    # EVERYTHING ELSE
    # -----------------------------------------------------

    return "other"


missing_df[
    "absence_category"
] = (
    missing_df[
        "reason"
    ]
    .apply(
        classify_absence_reason
    )
)


# ---------------------------------------------------------
# CHECK CATEGORY DISTRIBUTION
# ---------------------------------------------------------

print("=" * 80)
print("ABSENCE CATEGORY COUNTS")
print("=" * 80)

print(
    missing_df[
        "absence_category"
    ]
    .value_counts()
    .to_string()
)


print()

print("=" * 80)
print("PERCENT OF CONFIRMED ABSENCES")
print("=" * 80)

print(
    (
        missing_df[
            "absence_category"
        ]
        .value_counts(
            normalize=True
        )
        *
        100
    )
    .round(2)
    .to_string()
)


# ---------------------------------------------------------
# CHECK WHAT ENDED UP IN OTHER
# ---------------------------------------------------------

other_reasons = (
    missing_df[
        missing_df[
            "absence_category"
        ]
        ==
        "other"
    ][
        "reason"
    ]
    .value_counts()
)


print()

print("=" * 80)
print("REASONS CLASSIFIED AS OTHER")
print("=" * 80)


if len(other_reasons) == 0:

    print(
        "None — all reasons were classified."
    )

else:

    print(
        other_reasons.to_string()
    )


# ---------------------------------------------------------
# TEAM + FIXTURE CATEGORY COUNTS
# ---------------------------------------------------------

category_counts = (

    missing_df

    .groupby(
        [
            "fixture_id",
            "team",
            "absence_category"
        ]
    )

    .size()

    .unstack(
        fill_value=0
    )

    .reset_index()
)


# ---------------------------------------------------------
# EXPECTED CATEGORIES
# ---------------------------------------------------------

categories = [
    "physical_injury",
    "disciplinary",
    "illness_fitness",
    "selection_unavailable",
    "other"
]


for category in categories:

    if category not in category_counts.columns:

        category_counts[
            category
        ] = 0


# ---------------------------------------------------------
# LOAD OFFICIAL API-FOOTBALL FIXTURES
# ---------------------------------------------------------

fixtures = pd.read_csv(
    "../data/processed/api_football_pl_fixtures.csv"
)


print()

print("=" * 80)
print("FIXTURE FILE CHECK")
print("=" * 80)


print(
    "Fixture rows:",
    len(fixtures)
)


print(
    "Unique fixture IDs:",
    fixtures[
        "api_fixture_id"
    ].nunique()
)


print(
    "Duplicate fixture IDs:",
    fixtures[
        "api_fixture_id"
    ].duplicated().sum()
)


# ---------------------------------------------------------
# HOME ABSENCE FEATURES
#
# IMPORTANT:
# home_team_api is the original API-Football team name.
# That matches the team names in the injury endpoint.
# ---------------------------------------------------------

home_features = (

    fixtures[
        [
            "api_fixture_id",
            "home_team_api"
        ]
    ]

    .merge(

        category_counts,

        left_on=[
            "api_fixture_id",
            "home_team_api"
        ],

        right_on=[
            "fixture_id",
            "team"
        ],

        how="left"
    )
)


# ---------------------------------------------------------
# AWAY ABSENCE FEATURES
# ---------------------------------------------------------

away_features = (

    fixtures[
        [
            "api_fixture_id",
            "away_team_api"
        ]
    ]

    .merge(

        category_counts,

        left_on=[
            "api_fixture_id",
            "away_team_api"
        ],

        right_on=[
            "fixture_id",
            "team"
        ],

        how="left"
    )
)


# ---------------------------------------------------------
# CREATE FINAL MATCH-LEVEL DATASET
# ---------------------------------------------------------

absence_features = fixtures[
    [
        "api_fixture_id",
        "date",
        "api_season",
        "home_team",
        "away_team"
    ]
].copy()


# ---------------------------------------------------------
# CREATE HOME / AWAY / DIFFERENCE FEATURES
# ---------------------------------------------------------

for category in categories:

    absence_features[
        f"{category}_home"
    ] = (
        home_features[
            category
        ]
        .fillna(0)
        .astype(int)
    )


    absence_features[
        f"{category}_away"
    ] = (
        away_features[
            category
        ]
        .fillna(0)
        .astype(int)
    )


    absence_features[
        f"{category}_difference"
    ] = (
        absence_features[
            f"{category}_home"
        ]
        -
        absence_features[
            f"{category}_away"
        ]
    )


# ---------------------------------------------------------
# VALIDATION
# ---------------------------------------------------------

print()

print("=" * 80)
print("MATCH-LEVEL ABSENCE FEATURES")
print("=" * 80)


print(
    "Matches:",
    len(
        absence_features
    )
)


print(
    "Unique fixture IDs:",
    absence_features[
        "api_fixture_id"
    ].nunique()
)


print(
    "Duplicate fixture IDs:",
    absence_features[
        "api_fixture_id"
    ].duplicated().sum()
)


# ---------------------------------------------------------
# AVERAGE CATEGORY COUNTS
# ---------------------------------------------------------

print()

print("=" * 80)
print("AVERAGE ABSENCES PER MATCH SIDE")
print("=" * 80)


display_columns = []


for category in categories:

    display_columns.extend(
        [
            f"{category}_home",
            f"{category}_away"
        ]
    )


print(
    absence_features[
        display_columns
    ]
    .mean()
    .round(3)
    .to_string()
)


# ---------------------------------------------------------
# SANITY CHECK
#
# Every confirmed missing player should belong to exactly
# one category. Therefore the category totals should match
# the overall Missing Fixture counts we saw previously.
# ---------------------------------------------------------

absence_features[
    "category_total_home"
] = (
    absence_features[
        [
            f"{category}_home"
            for category in categories
        ]
    ]
    .sum(
        axis=1
    )
)


absence_features[
    "category_total_away"
] = (
    absence_features[
        [
            f"{category}_away"
            for category in categories
        ]
    ]
    .sum(
        axis=1
    )
)


print()

print("=" * 80)
print("CATEGORY TOTAL SANITY CHECK")
print("=" * 80)


print(
    "Average categorized missing — home:",
    round(
        absence_features[
            "category_total_home"
        ].mean(),
        3
    )
)


print(
    "Average categorized missing — away:",
    round(
        absence_features[
            "category_total_away"
        ].mean(),
        3
    )
)


print(
    "Total categorized missing — home:",
    int(
        absence_features[
            "category_total_home"
        ].sum()
    )
)


print(
    "Total categorized missing — away:",
    int(
        absence_features[
            "category_total_away"
        ].sum()
    )
)


print(
    "Combined categorized missing:",
    int(
        absence_features[
            "category_total_home"
        ].sum()
        +
        absence_features[
            "category_total_away"
        ].sum()
    )
)


print(
    "Original confirmed missing records:",
    len(
        missing_df
    )
)


# ---------------------------------------------------------
# REMOVE TEMPORARY SANITY-CHECK COLUMNS
# ---------------------------------------------------------

absence_features = (
    absence_features.drop(
        columns=[
            "category_total_home",
            "category_total_away"
        ]
    )
)


# ---------------------------------------------------------
# SAVE
# ---------------------------------------------------------

output_path = (
    "../data/processed/"
    "api_football_absence_type_features.csv"
)


absence_features.to_csv(
    output_path,
    index=False
)


print()

print(
    "Saved:",
    output_path
)

In [ ]:
# ---------------------------------------------------------
# API-FOOTBALL - STEP 19
# PLAYER CONTINUITY ACROSS SEASONS
# ---------------------------------------------------------

import json
import pandas as pd

from pathlib import Path


# ---------------------------------------------------------
# LOAD INJURY DATA
# ---------------------------------------------------------

injury_dir = Path(
    "../data/raw/api_football/injuries"
)

seasons = [
    2022,
    2023,
    2024
]

rows = []


for season in seasons:

    path = (
        injury_dir
        /
        f"premier_league_{season}_injuries.json"
    )


    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        data = json.load(f)


    for record in data.get(
        "response",
        []
    ):

        player = record.get(
            "player",
            {}
        )

        team = record.get(
            "team",
            {}
        )

        fixture = record.get(
            "fixture",
            {}
        )


        rows.append(
            {
                "season":
                    season,

                "fixture_id":
                    fixture.get("id"),

                "team_id":
                    team.get("id"),

                "team":
                    team.get("name"),

                "player_id":
                    player.get("id"),

                "player":
                    player.get("name"),

                "status":
                    player.get("type")
            }
        )


injury_df = pd.DataFrame(
    rows
)


# ---------------------------------------------------------
# CONFIRMED MISSING ONLY
# ---------------------------------------------------------

missing_df = (
    injury_df[
        injury_df["status"]
        ==
        "Missing Fixture"
    ]
    .copy()
)


# ---------------------------------------------------------
# UNIQUE PLAYER / SEASON COMBINATIONS
# ---------------------------------------------------------

player_seasons = (
    missing_df[
        [
            "player_id",
            "player",
            "season"
        ]
    ]
    .drop_duplicates()
)


# ---------------------------------------------------------
# PLAYERS APPEARING IN EACH SEASON'S INJURY DATA
# ---------------------------------------------------------

season_players = {}


for season in seasons:

    season_players[
        season
    ] = set(

        player_seasons.loc[
            player_seasons["season"]
            ==
            season,
            "player_id"
        ]
        .dropna()
        .astype(int)
    )


print("=" * 80)
print("UNIQUE CONFIRMED-MISSING PLAYERS BY SEASON")
print("=" * 80)


for season in seasons:

    print(
        season,
        ":",
        len(
            season_players[
                season
            ]
        )
    )


# ---------------------------------------------------------
# CONSECUTIVE-SEASON PLAYER OVERLAP
# ---------------------------------------------------------

print()

print("=" * 80)
print("CONSECUTIVE-SEASON PLAYER OVERLAP")
print("=" * 80)


overlap_22_23 = (
    season_players[2022]
    &
    season_players[2023]
)


overlap_23_24 = (
    season_players[2023]
    &
    season_players[2024]
)


print(
    "Players in both 2022 and 2023:",
    len(
        overlap_22_23
    )
)


print(
    "Players in both 2023 and 2024:",
    len(
        overlap_23_24
    )
)


print()

print(
    "2023 missing-player coverage from 2022:",
    round(
        len(
            overlap_22_23
        )
        /
        len(
            season_players[2023]
        ),
        3
    )
)


print(
    "2024 missing-player coverage from 2023:",
    round(
        len(
            overlap_23_24
        )
        /
        len(
            season_players[2024]
        ),
        3
    )
)


# ---------------------------------------------------------
# MORE IMPORTANT:
# WHAT PERCENTAGE OF ACTUAL MISSING-FIXTURE RECORDS
# INVOLVE A PLAYER SEEN IN THE PREVIOUS SEASON?
# ---------------------------------------------------------

missing_2023 = (
    missing_df[
        missing_df["season"]
        ==
        2023
    ]
    .copy()
)


missing_2024 = (
    missing_df[
        missing_df["season"]
        ==
        2024
    ]
    .copy()
)


missing_2023[
    "seen_previous_season"
] = (
    missing_2023[
        "player_id"
    ]
    .isin(
        season_players[2022]
    )
)


missing_2024[
    "seen_previous_season"
] = (
    missing_2024[
        "player_id"
    ]
    .isin(
        season_players[2023]
    )
)


print()

print("=" * 80)
print("MISSING-FIXTURE RECORD COVERAGE")
print("=" * 80)


print(
    "2023 total confirmed absence records:",
    len(
        missing_2023
    )
)


print(
    "2023 records involving player seen in 2022:",
    int(
        missing_2023[
            "seen_previous_season"
        ].sum()
    )
)


print(
    "2023 record coverage:",
    round(
        missing_2023[
            "seen_previous_season"
        ].mean(),
        3
    )
)


print()


print(
    "2024 total confirmed absence records:",
    len(
        missing_2024
    )
)


print(
    "2024 records involving player seen in 2023:",
    int(
        missing_2024[
            "seen_previous_season"
        ].sum()
    )
)


print(
    "2024 record coverage:",
    round(
        missing_2024[
            "seen_previous_season"
        ].mean(),
        3
    )
)


# ---------------------------------------------------------
# PLAYERS PRESENT IN ALL THREE SEASONS
# ---------------------------------------------------------

all_three = (
    season_players[2022]
    &
    season_players[2023]
    &
    season_players[2024]
)


print()

print("=" * 80)
print("LONG-TERM CONTINUITY")
print("=" * 80)


print(
    "Players appearing in all 3 seasons:",
    len(
        all_three
    )
)

In [ ]:
# ---------------------------------------------------------
# API-FOOTBALL - STEP 20
# INSPECT PLAYER-SEASON ENDPOINT FOR TARGETED COLLECTION
# ---------------------------------------------------------

import requests
import pandas as pd


# ---------------------------------------------------------
# REQUEST ONE PAGE ONLY
#
# 2023 means the 2023/24 Premier League season.
# We would potentially use these stats to measure player
# importance going into 2024/25.
# ---------------------------------------------------------

url = (
    f"{BASE_URL}/players"
)


params = {
    "league": PREMIER_LEAGUE_ID,
    "season": 2023,
    "page": 1
}


response = requests.get(
    url,
    headers=HEADERS,
    params=params
)


print("=" * 80)
print("REQUEST CHECK")
print("=" * 80)

print(
    "HTTP status:",
    response.status_code
)

print(
    "Requests remaining:",
    response.headers.get(
        "x-ratelimit-requests-remaining"
    )
)


data = response.json()


print()

print("=" * 80)
print("PAGING")
print("=" * 80)

print(
    data.get(
        "paging"
    )
)


print(
    "Results on this page:",
    data.get(
        "results"
    )
)


# ---------------------------------------------------------
# INSPECT FIRST PLAYER
# ---------------------------------------------------------

players = data.get(
    "response",
    []
)


if len(players) > 0:

    first = players[0]

    print()

    print("=" * 80)
    print("TOP-LEVEL PLAYER RECORD KEYS")
    print("=" * 80)

    print(
        first.keys()
    )


    print()

    print("=" * 80)
    print("PLAYER OBJECT")
    print("=" * 80)

    print(
        first.get(
            "player"
        )
    )


    print()

    print("=" * 80)
    print("STATISTICS ENTRIES")
    print("=" * 80)

    print(
        "Number of statistics entries:",
        len(
            first.get(
                "statistics",
                []
            )
        )
    )


    for i, stats in enumerate(
        first.get(
            "statistics",
            []
        )
    ):

        print()

        print(
            f"STATISTICS ENTRY {i + 1}"
        )

        print(
            "Team:",
            stats.get(
                "team"
            )
        )

        print(
            "League:",
            stats.get(
                "league"
            )
        )

        print(
            "Games:",
            stats.get(
                "games"
            )
        )

        print(
            "Goals:",
            stats.get(
                "goals"
            )
        )

        print(
            "Passes:",
            stats.get(
                "passes"
            )
        )

        print(
            "Tackles:",
            stats.get(
                "tackles"
            )
        )


# ---------------------------------------------------------
# CREATE SMALL PAGE SUMMARY
# ---------------------------------------------------------

summary_rows = []


for record in players:

    player = record.get(
        "player",
        {}
    )


    for stats in record.get(
        "statistics",
        []
    ):

        games = stats.get(
            "games",
            {}
        )

        goals = stats.get(
            "goals",
            {}
        )

        team = stats.get(
            "team",
            {}
        )


        summary_rows.append(
            {
                "player_id":
                    player.get("id"),

                "player":
                    player.get("name"),

                "team":
                    team.get("name"),

                "appearances":
                    games.get("appearences"),

                "starts":
                    games.get("lineups"),

                "minutes":
                    games.get("minutes"),

                "rating":
                    games.get("rating"),

                "goals":
                    goals.get("total"),

                "assists":
                    goals.get("assists")
            }
        )


page_summary = pd.DataFrame(
    summary_rows
)


print()

print("=" * 80)
print("PAGE 1 PLAYER SUMMARY")
print("=" * 80)

print(
    page_summary.to_string(
        index=False
    )
)

In [ ]:
# ---------------------------------------------------------
# API-FOOTBALL - STEP 21
# TEST TARGETED PLAYER-SEASON QUERY
# ---------------------------------------------------------

import requests
import json


# ---------------------------------------------------------
# TEST PLAYER
#
# Rodri:
# player_id = 44
#
# We already saw him on the full 2023/24 season query,
# so we know what the response should roughly look like.
# ---------------------------------------------------------

test_player_id = 44


url = (
    f"{BASE_URL}/players"
)


params = {
    "id": test_player_id,
    "league": PREMIER_LEAGUE_ID,
    "season": 2023
}


response = requests.get(
    url,
    headers=HEADERS,
    params=params
)


print("=" * 80)
print("REQUEST CHECK")
print("=" * 80)

print(
    "HTTP status:",
    response.status_code
)

print(
    "Requests remaining:",
    response.headers.get(
        "x-ratelimit-requests-remaining"
    )
)


data = response.json()


print()

print("=" * 80)
print("API RESPONSE SUMMARY")
print("=" * 80)

print(
    "Errors:",
    data.get("errors")
)

print(
    "Results:",
    data.get("results")
)

print(
    "Paging:",
    data.get("paging")
)


# ---------------------------------------------------------
# INSPECT RETURNED PLAYER
# ---------------------------------------------------------

players = data.get(
    "response",
    []
)


if len(players) == 0:

    print()
    print(
        "No player data returned."
    )


else:

    print()

    print("=" * 80)
    print("TARGETED PLAYER RESULT")
    print("=" * 80)


    for record in players:

        player = record.get(
            "player",
            {}
        )


        print(
            "Player ID:",
            player.get("id")
        )

        print(
            "Player:",
            player.get("name")
        )


        for stats in record.get(
            "statistics",
            []
        ):

            team = stats.get(
                "team",
                {}
            )

            games = stats.get(
                "games",
                {}
            )

            goals = stats.get(
                "goals",
                {}
            )

            passes = stats.get(
                "passes",
                {}
            )

            tackles = stats.get(
                "tackles",
                {}
            )


            print()

            print(
                "Team:",
                team.get("name")
            )

            print(
                "Position:",
                games.get("position")
            )

            print(
                "Appearances:",
                games.get("appearences")
            )

            print(
                "Starts:",
                games.get("lineups")
            )

            print(
                "Minutes:",
                games.get("minutes")
            )

            print(
                "Rating:",
                games.get("rating")
            )

            print(
                "Goals:",
                goals.get("total")
            )

            print(
                "Assists:",
                goals.get("assists")
            )

            print(
                "Passes:",
                passes.get("total")
            )

            print(
                "Key passes:",
                passes.get("key")
            )

            print(
                "Tackles:",
                tackles.get("total")
            )

            print(
                "Interceptions:",
                tackles.get("interceptions")
            )


# ---------------------------------------------------------
# RAW STATISTICS STRUCTURE
#
# Useful in case there are fields we want later.
# ---------------------------------------------------------

if len(players) > 0:

    print()

    print("=" * 80)
    print("FULL STATISTICS OBJECT")
    print("=" * 80)

    print(
        json.dumps(
            players[0].get(
                "statistics",
                []
            ),
            indent=2
        )
    )

In [ ]:
# ---------------------------------------------------------
# API-FOOTBALL - STEP 22
# TARGETED VS BULK PLAYER COLLECTION ANALYSIS
# ---------------------------------------------------------

import json
import pandas as pd

from pathlib import Path


# ---------------------------------------------------------
# LOAD CONFIRMED ABSENCE DATA
# ---------------------------------------------------------

injury_dir = Path(
    "../data/raw/api_football/injuries"
)

seasons = [
    2023,
    2024
]

rows = []


for season in seasons:

    path = (
        injury_dir
        /
        f"premier_league_{season}_injuries.json"
    )

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        data = json.load(f)


    for record in data.get(
        "response",
        []
    ):

        player = record.get(
            "player",
            {}
        )

        team = record.get(
            "team",
            {}
        )


        if (
            player.get("type")
            ==
            "Missing Fixture"
        ):

            rows.append(
                {
                    "season":
                        season,

                    "player_id":
                        player.get("id"),

                    "player":
                        player.get("name"),

                    "team_id":
                        team.get("id"),

                    "team":
                        team.get("name")
                }
            )


missing_df = pd.DataFrame(
    rows
)


# ---------------------------------------------------------
# REMOVE RECORDS WITHOUT PLAYER IDs
# ---------------------------------------------------------

missing_df = (
    missing_df[
        missing_df[
            "player_id"
        ].notna()
    ]
    .copy()
)


missing_df[
    "player_id"
] = (
    missing_df[
        "player_id"
    ]
    .astype(int)
)


# ---------------------------------------------------------
# COVERAGE TARGETS
# ---------------------------------------------------------

coverage_targets = [
    0.50,
    0.70,
    0.80,
    0.90,
    0.95
]


summary_rows = []


# ---------------------------------------------------------
# ANALYZE EACH SEASON
# ---------------------------------------------------------

for season in seasons:

    season_df = (
        missing_df[
            missing_df[
                "season"
            ]
            ==
            season
        ]
        .copy()
    )


    # -----------------------------------------------------
    # COUNT ABSENCE RECORDS PER PLAYER
    # -----------------------------------------------------

    player_counts = (

        season_df

        .groupby(
            [
                "player_id",
                "player"
            ]
        )

        .size()

        .reset_index(
            name="absence_records"
        )

        .sort_values(
            "absence_records",
            ascending=False
        )

        .reset_index(
            drop=True
        )
    )


    total_records = (
        player_counts[
            "absence_records"
        ]
        .sum()
    )


    player_counts[
        "cumulative_records"
    ] = (
        player_counts[
            "absence_records"
        ]
        .cumsum()
    )


    player_counts[
        "cumulative_coverage"
    ] = (
        player_counts[
            "cumulative_records"
        ]
        /
        total_records
    )


    print()
    print("=" * 80)
    print(
        f"SEASON {season}"
    )
    print("=" * 80)


    print(
        "Confirmed absence records:",
        total_records
    )


    print(
        "Unique missing players:",
        player_counts[
            "player_id"
        ].nunique()
    )


    print()

    print(
        "TOP 20 PLAYERS BY ABSENCE RECORDS"
    )


    print(
        player_counts[
            [
                "player_id",
                "player",
                "absence_records",
                "cumulative_coverage"
            ]
        ]
        .head(20)
        .round(
            {
                "cumulative_coverage":
                    3
            }
        )
        .to_string(
            index=False
        )
    )


    print()

    print(
        "COVERAGE REQUIREMENTS"
    )


    # -----------------------------------------------------
    # NUMBER OF PLAYERS NEEDED FOR EACH COVERAGE TARGET
    # -----------------------------------------------------

    for target in coverage_targets:

        players_needed = (
            (
                player_counts[
                    "cumulative_coverage"
                ]
                <
                target
            )
            .sum()
            +
            1
        )


        actual_coverage = (
            player_counts.iloc[
                players_needed - 1
            ][
                "cumulative_coverage"
            ]
        )


        print(
            f"{int(target * 100)}% coverage:",
            players_needed,
            "players",
            "| actual:",
            round(
                actual_coverage,
                3
            )
        )


        summary_rows.append(
            {
                "season":
                    season,

                "coverage_target":
                    target,

                "players_needed":
                    players_needed,

                "actual_coverage":
                    actual_coverage
            }
        )


# ---------------------------------------------------------
# FINAL SUMMARY
# ---------------------------------------------------------

coverage_summary = pd.DataFrame(
    summary_rows
)


print()
print("=" * 80)
print("TARGETED COLLECTION SUMMARY")
print("=" * 80)


print(
    coverage_summary
    .round(3)
    .to_string(
        index=False
    )
)


# ---------------------------------------------------------
# BULK BREAK-EVEN
#
# 2023/24 currently requires 51 pages.
# One targeted player = one request.
#
# So if the player count is above ~51, bulk collection
# is already cheaper for that season.
# ---------------------------------------------------------

print()
print("=" * 80)
print("COLLECTION STRATEGY")
print("=" * 80)


bulk_requests_2023 = 51


print(
    "2023 bulk season requests:",
    bulk_requests_2023
)


print(
    "Targeted break-even:",
    bulk_requests_2023,
    "players"
)


print()

print(
    "If desired coverage requires fewer than",
    bulk_requests_2023,
    "players, targeted queries may be cheaper."
)


print(
    "If it requires more than",
    bulk_requests_2023,
    "players, bulk season collection is cheaper."
)

In [ ]:
# ---------------------------------------------------------
# API-FOOTBALL - STEP 23
# SAME-CLUB CONTINUITY FOR MISSING PLAYERS
# ---------------------------------------------------------

import json
import pandas as pd

from pathlib import Path


# ---------------------------------------------------------
# LOAD CONFIRMED ABSENCES
# ---------------------------------------------------------

injury_dir = Path(
    "../data/raw/api_football/injuries"
)

rows = []


for season in [2022, 2023, 2024]:

    path = (
        injury_dir
        /
        f"premier_league_{season}_injuries.json"
    )

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        data = json.load(f)


    for record in data.get(
        "response",
        []
    ):

        player = record.get(
            "player",
            {}
        )

        team = record.get(
            "team",
            {}
        )


        if (
            player.get("type")
            ==
            "Missing Fixture"
        ):

            rows.append(
                {
                    "season":
                        season,

                    "player_id":
                        player.get("id"),

                    "player":
                        player.get("name"),

                    "team_id":
                        team.get("id"),

                    "team":
                        team.get("name")
                }
            )


missing_df = pd.DataFrame(
    rows
)


# ---------------------------------------------------------
# REMOVE NULL PLAYER / TEAM IDS
# ---------------------------------------------------------

missing_df = (
    missing_df[
        missing_df["player_id"].notna()
        &
        missing_df["team_id"].notna()
    ]
    .copy()
)


missing_df["player_id"] = (
    missing_df["player_id"]
    .astype(int)
)


missing_df["team_id"] = (
    missing_df["team_id"]
    .astype(int)
)


# ---------------------------------------------------------
# UNIQUE PLAYER-TEAM COMBINATIONS BY SEASON
# ---------------------------------------------------------

player_team_seasons = (

    missing_df[
        [
            "season",
            "player_id",
            "player",
            "team_id",
            "team"
        ]
    ]

    .drop_duplicates()
)


# ---------------------------------------------------------
# TEST 2023 ABSENCES AGAINST 2022
# ---------------------------------------------------------

previous_2022 = (

    player_team_seasons[
        player_team_seasons["season"]
        ==
        2022
    ][
        [
            "player_id",
            "team_id"
        ]
    ]

    .drop_duplicates()
)


current_2023 = (

    missing_df[
        missing_df["season"]
        ==
        2023
    ]

    .copy()
)


previous_2022_pairs = set(
    zip(
        previous_2022["player_id"],
        previous_2022["team_id"]
    )
)


current_2023[
    "same_player_team_previous"
] = [

    (
        player_id,
        team_id
    )
    in
    previous_2022_pairs

    for player_id, team_id
    in zip(
        current_2023["player_id"],
        current_2023["team_id"]
    )
]


# ---------------------------------------------------------
# TEST 2024 ABSENCES AGAINST 2023
# ---------------------------------------------------------

previous_2023 = (

    player_team_seasons[
        player_team_seasons["season"]
        ==
        2023
    ][
        [
            "player_id",
            "team_id"
        ]
    ]

    .drop_duplicates()
)


current_2024 = (

    missing_df[
        missing_df["season"]
        ==
        2024
    ]

    .copy()
)


previous_2023_pairs = set(
    zip(
        previous_2023["player_id"],
        previous_2023["team_id"]
    )
)


current_2024[
    "same_player_team_previous"
] = [

    (
        player_id,
        team_id
    )
    in
    previous_2023_pairs

    for player_id, team_id
    in zip(
        current_2024["player_id"],
        current_2024["team_id"]
    )
]


# ---------------------------------------------------------
# PLAYER-LEVEL CONTINUITY
# ---------------------------------------------------------

unique_2023 = (

    current_2023[
        [
            "player_id",
            "team_id",
            "same_player_team_previous"
        ]
    ]

    .drop_duplicates()
)


unique_2024 = (

    current_2024[
        [
            "player_id",
            "team_id",
            "same_player_team_previous"
        ]
    ]

    .drop_duplicates()
)


# ---------------------------------------------------------
# PRINT RESULTS
# ---------------------------------------------------------

print("=" * 80)
print("2023/24 SAME-CLUB CONTINUITY")
print("=" * 80)


print(
    "Confirmed absence records:",
    len(current_2023)
)


print(
    "Absence records with same player/team seen in 2022/23:",
    int(
        current_2023[
            "same_player_team_previous"
        ].sum()
    )
)


print(
    "Record coverage:",
    round(
        current_2023[
            "same_player_team_previous"
        ].mean(),
        3
    )
)


print()

print(
    "Unique player-team combinations:",
    len(unique_2023)
)


print(
    "Same player/team seen previous season:",
    int(
        unique_2023[
            "same_player_team_previous"
        ].sum()
    )
)


print(
    "Player-team coverage:",
    round(
        unique_2023[
            "same_player_team_previous"
        ].mean(),
        3
    )
)


print()
print("=" * 80)
print("2024/25 SAME-CLUB CONTINUITY")
print("=" * 80)


print(
    "Confirmed absence records:",
    len(current_2024)
)


print(
    "Absence records with same player/team seen in 2023/24:",
    int(
        current_2024[
            "same_player_team_previous"
        ].sum()
    )
)


print(
    "Record coverage:",
    round(
        current_2024[
            "same_player_team_previous"
        ].mean(),
        3
    )
)


print()

print(
    "Unique player-team combinations:",
    len(unique_2024)
)


print(
    "Same player/team seen previous season:",
    int(
        unique_2024[
            "same_player_team_previous"
        ].sum()
    )
)


print(
    "Player-team coverage:",
    round(
        unique_2024[
            "same_player_team_previous"
        ].mean(),
        3
    )
)


# ---------------------------------------------------------
# COMPARE PLAYER-ONLY VS PLAYER+TEAM CONTINUITY
# ---------------------------------------------------------

players_2022 = set(
    player_team_seasons.loc[
        player_team_seasons["season"]
        ==
        2022,
        "player_id"
    ]
)


players_2023 = set(
    player_team_seasons.loc[
        player_team_seasons["season"]
        ==
        2023,
        "player_id"
    ]
)


current_2023[
    "player_seen_previous"
] = (
    current_2023["player_id"]
    .isin(players_2022)
)


current_2024[
    "player_seen_previous"
] = (
    current_2024["player_id"]
    .isin(players_2023)
)


print()
print("=" * 80)
print("PLAYER-ONLY VS SAME-CLUB RECORD COVERAGE")
print("=" * 80)


print(
    "2023/24 player-only:",
    round(
        current_2023[
            "player_seen_previous"
        ].mean(),
        3
    )
)


print(
    "2023/24 same-club:",
    round(
        current_2023[
            "same_player_team_previous"
        ].mean(),
        3
    )
)


print()


print(
    "2024/25 player-only:",
    round(
        current_2024[
            "player_seen_previous"
        ].mean(),
        3
    )
)


print(
    "2024/25 same-club:",
    round(
        current_2024[
            "same_player_team_previous"
        ].mean(),
        3
    )
)

In [ ]:
# =========================================================
# STEP 24 — RESUMABLE API-FOOTBALL PLAYER STAT COLLECTION
# =========================================================

import requests
import json
import time
from pathlib import Path


# ---------------------------------------------------------
# SETTINGS
# ---------------------------------------------------------

LEAGUE_ID = 39  # Premier League

# Previous-season player stats needed for our experiments:
# 2022 -> used for 2023/24 prediction
# 2023 -> used for 2024/25 prediction
SEASONS = [2022, 2023]

# Keep a few requests available in case we need debugging
MIN_REQUESTS_REMAINING = 3

# Small delay between calls
REQUEST_DELAY = 0.25


# ---------------------------------------------------------
# SAVE LOCATION
# ---------------------------------------------------------

PLAYER_DIR = Path(
    "../data/raw/api_football/players"
)

PLAYER_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Saving player pages to:")
print(PLAYER_DIR.resolve())


# ---------------------------------------------------------
# HELPER — CHECK WHETHER PAGE ALREADY EXISTS
# ---------------------------------------------------------

def page_file(season, page):

    return PLAYER_DIR / (
        f"premier_league_{season}_players_page_{page}.json"
    )


# ---------------------------------------------------------
# HELPER — API REQUEST
# ---------------------------------------------------------

def request_player_page(season, page):

    url = f"{BASE_URL}/players"

    params = {
        "league": LEAGUE_ID,
        "season": season,
        "page": page
    }

    response = requests.get(
        url,
        headers=HEADERS,
        params=params,
        timeout=30
    )

    remaining = response.headers.get(
        "x-ratelimit-requests-remaining"
    )

    try:
        remaining = int(remaining)
    except (TypeError, ValueError):
        remaining = None

    return response, remaining


# ---------------------------------------------------------
# COLLECTION
# ---------------------------------------------------------

stop_collection = False

for season in SEASONS:

    if stop_collection:
        break

    print("\n" + "=" * 80)
    print(f"COLLECTING PLAYER STATS — SEASON {season}")
    print("=" * 80)

    # -----------------------------------------------------
    # Determine first page we still need
    # -----------------------------------------------------

    page = 1

    while page_file(season, page).exists():

        print(
            f"Season {season} | Page {page} already saved — skipping."
        )

        page += 1

    print()
    print("Starting/resuming at page:", page)

    # -----------------------------------------------------
    # Download pages
    # -----------------------------------------------------

    while True:

        print("\n" + "-" * 80)

        print(
            f"Season {season} | Requesting page {page}"
        )

        try:

            response, remaining = request_player_page(
                season,
                page
            )

        except requests.RequestException as e:

            print("REQUEST FAILED:")
            print(e)

            print(
                "\nStopping safely. "
                "Rerun this cell later to resume."
            )

            stop_collection = True
            break

        print(
            "HTTP status:",
            response.status_code
        )

        print(
            "Requests remaining:",
            remaining
        )

        # -------------------------------------------------
        # HTTP failure
        # -------------------------------------------------

        if response.status_code != 200:

            print(
                "Non-200 response received."
            )

            print(response.text[:1000])

            print(
                "\nStopping safely."
            )

            stop_collection = True
            break

        # -------------------------------------------------
        # Parse JSON
        # -------------------------------------------------

        data = response.json()

        errors = data.get(
            "errors",
            []
        )

        print(
            "Errors:",
            errors
        )

        if errors:

            print(
                "\nAPI returned an error. "
                "Stopping safely."
            )

            stop_collection = True
            break

        # -------------------------------------------------
        # Inspect page
        # -------------------------------------------------

        results = data.get(
            "response",
            []
        )

        paging = data.get(
            "paging",
            {}
        )

        current_page = paging.get(
            "current"
        )

        total_pages = paging.get(
            "total"
        )

        print(
            "Results:",
            len(results)
        )

        print(
            "Paging:",
            paging
        )

        # -------------------------------------------------
        # Save immediately
        # -------------------------------------------------

        output_file = page_file(
            season,
            page
        )

        with open(
            output_file,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                data,
                f,
                ensure_ascii=False,
                indent=2
            )

        print(
            "Saved:",
            output_file
        )

        # -------------------------------------------------
        # Finished season
        # -------------------------------------------------

        if (
            total_pages is not None
            and current_page is not None
            and current_page >= total_pages
        ):

            print("\n" + "=" * 80)

            print(
                f"SEASON {season} COMPLETE"
            )

            print(
                f"Total pages: {total_pages}"
            )

            print("=" * 80)

            break

        # -------------------------------------------------
        # Protect remaining quota
        # -------------------------------------------------

        if (
            remaining is not None
            and remaining <= MIN_REQUESTS_REMAINING
        ):

            print("\n" + "=" * 80)

            print(
                "DAILY REQUEST SAFETY LIMIT REACHED"
            )

            print(
                "Requests remaining:",
                remaining
            )

            print(
                "Collection has been saved."
            )

            print(
                "Rerun this exact cell after the quota resets."
            )

            print("=" * 80)

            stop_collection = True
            break

        # -------------------------------------------------
        # Next page
        # -------------------------------------------------

        page += 1

        time.sleep(
            REQUEST_DELAY
        )


# ---------------------------------------------------------
# COLLECTION SUMMARY
# ---------------------------------------------------------

print("\n\n" + "=" * 80)
print("CURRENT COLLECTION STATUS")
print("=" * 80)

for season in SEASONS:

    files = sorted(
        PLAYER_DIR.glob(
            f"premier_league_{season}_players_page_*.json"
        )
    )

    print(
        f"{season}: {len(files)} pages saved"
    )

print("\nDone.")

# Picking up where we left off on new laptop

In [ ]:
import os
import requests
from dotenv import load_dotenv

load_dotenv("../.env")

API_FOOTBALL_KEY = os.getenv("API_FOOTBALL_KEY")
API_FOOTBALL_BASE_URL = "https://v3.football.api-sports.io"

headers = {
    "x-apisports-key": API_FOOTBALL_KEY
}

url = f"{API_FOOTBALL_BASE_URL}/players"

params = {
    "league": 39,
    "season": 2022,
    "page": 4
}

response = requests.get(
    url,
    headers=headers,
    params=params,
    timeout=30
)

data = response.json()

print("HTTP status:", response.status_code)
print("Errors:", data.get("errors"))
print("Results returned:", data.get("results"))
print("Paging:", data.get("paging"))

In [ ]:
# Collect remaining pages

import json
import time
from pathlib import Path

PLAYER_RAW_DIR = Path("../data/raw/api_football/players")
PLAYER_RAW_DIR.mkdir(parents=True, exist_ok=True)

league_id = 39
season = 2022
total_pages = 45

for page in range(1, total_pages + 1):

    save_path = PLAYER_RAW_DIR / f"premier_league_{season}_players_page_{page}.json"

    # Skip pages we already downloaded
    if save_path.exists():
        print(f"Page {page}: already saved — skipping")
        continue

    params = {
        "league": league_id,
        "season": season,
        "page": page
    }

    response = requests.get(
        f"{API_FOOTBALL_BASE_URL}/players",
        headers=headers,
        params=params,
        timeout=30
    )

    data = response.json()

    if response.status_code != 200 or data.get("errors"):
        print(f"Page {page}: ERROR")
        print(data.get("errors"))
        break

    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)

    print(
        f"Page {page}: saved "
        f"({data.get('results')} players)"
    )

    # Small pause so we do not unnecessarily hammer the API
    time.sleep(0.25)

print("Collection finished.")

In [ ]:
# Make sure we didn't miss a page

from pathlib import Path

PLAYER_RAW_DIR = Path("../data/raw/api_football/players")

season = 2022

saved_pages = sorted(
    PLAYER_RAW_DIR.glob(f"premier_league_{season}_players_page_*.json")
)

print("Saved files:", len(saved_pages))

page_numbers = sorted(
    int(path.stem.split("_")[-1])
    for path in saved_pages
)

expected_pages = list(range(1, 46))

missing_pages = sorted(set(expected_pages) - set(page_numbers))

print("First page:", page_numbers[0])
print("Last page:", page_numbers[-1])
print("Missing pages:", missing_pages)


In [ ]:
# Collect 2023 seasonal information

season = 2023
total_pages = 51

for page in range(1, total_pages + 1):

    save_path = PLAYER_RAW_DIR / f"premier_league_{season}_players_page_{page}.json"

    # Skip anything already saved
    if save_path.exists():
        print(f"Page {page}: already saved — skipping")
        continue

    params = {
        "league": league_id,
        "season": season,
        "page": page
    }

    response = requests.get(
        f"{API_FOOTBALL_BASE_URL}/players",
        headers=headers,
        params=params,
        timeout=30
    )

    data = response.json()

    if response.status_code != 200 or data.get("errors"):
        print(f"Page {page}: ERROR")
        print(data.get("errors"))
        break

    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)

    print(
        f"Page {page}: saved "
        f"({data.get('results')} players)"
    )

    time.sleep(0.25)

print("Collection finished.")

In [ ]:
season = 2023

saved_pages = sorted(
    PLAYER_RAW_DIR.glob(f"premier_league_{season}_players_page_*.json")
)

page_numbers = sorted(
    int(path.stem.split("_")[-1])
    for path in saved_pages
)

expected_pages = list(range(1, 52))

missing_pages = sorted(set(expected_pages) - set(page_numbers))

print("Saved files:", len(saved_pages))
print("First page:", page_numbers[0])
print("Last page:", page_numbers[-1])
print("Missing pages:", missing_pages)

In [ ]:
import json
from pathlib import Path

sample_file = Path(
    "../data/raw/api_football/players/"
    "premier_league_2023_players_page_1.json"
)

with open(sample_file, "r", encoding="utf-8") as f:
    sample_data = json.load(f)

print("Top-level keys:")
print(sample_data.keys())

print("\nNumber of player records:")
print(len(sample_data["response"]))

first_player = sample_data["response"][0]

print("\nFirst player top-level keys:")
print(first_player.keys())

print("\nPlayer info:")
print(first_player["player"])

print("\nNumber of statistics entries:")
print(len(first_player["statistics"]))

print("\nFirst statistics entry:")
print(first_player["statistics"][0])

In [ ]:
import json
from pathlib import Path

PLAYER_RAW_DIR = Path("../data/raw/api_football/players")

for season in [2022, 2023]:

    files = sorted(
        PLAYER_RAW_DIR.glob(
            f"premier_league_{season}_players_page_*.json"
        )
    )

    total_players = 0
    multi_team_players = []

    for file_path in files:

        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        for record in data["response"]:

            total_players += 1

            player = record["player"]
            statistics = record["statistics"]

            team_names = list({
                stat["team"]["name"]
                for stat in statistics
            })

            if len(team_names) > 1:
                multi_team_players.append({
                    "player_id": player["id"],
                    "player_name": player["name"],
                    "teams": team_names,
                    "statistics_entries": len(statistics)
                })

    print(f"\nSeason {season}")
    print("Total players:", total_players)
    print("Players with multiple teams:", len(multi_team_players))

    print("\nExamples:")
    for player in multi_team_players[:10]:
        print(player)

In [ ]:
import json
import pandas as pd
from pathlib import Path

PLAYER_RAW_DIR = Path("../data/raw/api_football/players")

player_rows = []

for season in [2022, 2023]:

    files = sorted(
        PLAYER_RAW_DIR.glob(
            f"premier_league_{season}_players_page_*.json"
        )
    )

    for file_path in files:

        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        for record in data["response"]:

            player = record["player"]

            for stat in record["statistics"]:

                games = stat.get("games", {})
                goals = stat.get("goals", {})

                player_rows.append({
                    "season": season,

                    "player_id": player.get("id"),
                    "player_name": player.get("name"),

                    "team_id": stat["team"].get("id"),
                    "team_name": stat["team"].get("name"),

                    "position": games.get("position"),

                    "appearances": games.get("appearences"),
                    "starts": games.get("lineups"),
                    "minutes": games.get("minutes"),

                    "goals": goals.get("total"),
                    "assists": goals.get("assists")
                })

player_season_team = pd.DataFrame(player_rows)

player_season_team = (
    player_season_team
    .sort_values(
        ["season", "team_name", "minutes"],
        ascending=[True, True, False]
    )
    .reset_index(drop=True)
)

print("Shape:", player_season_team.shape)

print("\nRows by season:")
print(
    player_season_team
    .groupby("season")
    .size()
)

print("\nMissing minutes:")
print(player_season_team["minutes"].isna().sum())

print("\nSample:")
display(player_season_team.head(10))

In [ ]:
# Inspect the missing minutes column

missing_minutes = player_season_team[
    player_season_team["minutes"].isna()
].copy()

print("Total missing-minute rows:", len(missing_minutes))

print("\nMissing-minute rows by season:")
print(
    missing_minutes
    .groupby("season")
    .size()
)

print("\nAppearances among missing-minute rows:")
print(
    missing_minutes["appearances"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nStarts among missing-minute rows:")
print(
    missing_minutes["starts"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nSample missing-minute players:")
display(
    missing_minutes[
        [
            "season",
            "player_id",
            "player_name",
            "team_name",
            "position",
            "appearances",
            "starts",
            "minutes"
        ]
    ].head(20)
)

In [ ]:
odd_missing_minutes = player_season_team[
    player_season_team["minutes"].isna()
    & player_season_team["appearances"].notna()
]

print("Player with appearances but missing minutes:")
display(odd_missing_minutes)

if len(odd_missing_minutes) > 0:

    odd_player_ids = odd_missing_minutes["player_id"].unique()

    print("\nAll rows for this player:")
    display(
        player_season_team[
            player_season_team["player_id"].isin(odd_player_ids)
        ].sort_values(["season", "player_id", "team_name"])
    )

In [ ]:
valid_minutes = player_season_team[
    player_season_team["minutes"].notna()
].copy()

print("Valid player-team rows:", len(valid_minutes))

print("\nMinutes summary:")
print(
    valid_minutes["minutes"].describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90, 0.95]
    )
)

print("\nMinutes by season:")
print(
    valid_minutes
    .groupby("season")["minutes"]
    .agg(["count", "mean", "median", "max"])
)

print("\nTop 10 player-team seasons by minutes:")
display(
    valid_minutes[
        [
            "season",
            "player_name",
            "team_name",
            "position",
            "appearances",
            "starts",
            "minutes"
        ]
    ]
    .sort_values("minutes", ascending=False)
    .head(10)
)

In [ ]:
MAX_PL_MINUTES = 38 * 90

player_season_team["minutes_importance"] = (
    player_season_team["minutes"] / MAX_PL_MINUTES
).clip(lower=0, upper=1)

print("Maximum PL minutes:", MAX_PL_MINUTES)

print("\nImportance summary:")
print(
    player_season_team["minutes_importance"].describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90, 0.95]
    )
)

print("\nMissing importance values:")
print(
    player_season_team["minutes_importance"].isna().sum()
)

print("\nExample importance levels:")
display(
    player_season_team[
        player_season_team["minutes"].notna()
    ][
        [
            "season",
            "player_name",
            "team_name",
            "minutes",
            "minutes_importance"
        ]
    ]
    .sort_values("minutes", ascending=False)
    .head(15)
)

In [ ]:
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")

print("Processed files related to injuries/API-Football:")

for path in sorted(PROCESSED_DIR.glob("*")):
    name = path.name.lower()

    if "injur" in name or "api" in name:
        print(path.name)

In [ ]:
import pandas as pd
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")

missing_player_summary = pd.read_csv(
    PROCESSED_DIR / "api_football_missing_player_summary.csv"
)

team_fixture_injuries = pd.read_csv(
    PROCESSED_DIR / "api_football_team_fixture_injuries.csv"
)

print("MISSING PLAYER SUMMARY")
print("Shape:", missing_player_summary.shape)
print("Columns:")
print(missing_player_summary.columns.tolist())

print("\nSample:")
display(missing_player_summary.head(10))


print("\n" + "=" * 80)


print("\nTEAM-FIXTURE INJURIES")
print("Shape:", team_fixture_injuries.shape)
print("Columns:")
print(team_fixture_injuries.columns.tolist())

print("\nSample:")
display(team_fixture_injuries.head(10))

In [ ]:
import json
from pathlib import Path

INJURY_RAW_DIR = Path("../data/raw/api_football/injuries")

injury_files = sorted(
    INJURY_RAW_DIR.glob("premier_league_*_injuries.json")
)

print("Injury files found:")
for path in injury_files:
    print(path.name)

with open(injury_files[0], "r", encoding="utf-8") as f:
    sample_injuries = json.load(f)

print("\nTop-level keys:")
print(sample_injuries.keys())

print("\nNumber of records:")
print(len(sample_injuries["response"]))

print("\nFirst raw injury record:")
print(sample_injuries["response"][0])

In [ ]:
injury_rows = []

for file_path in injury_files:

    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    for record in data["response"]:

        player = record.get("player", {})
        team = record.get("team", {})
        fixture = record.get("fixture", {})
        league = record.get("league", {})

        injury_rows.append({
            "fixture_id": fixture.get("id"),
            "fixture_date": fixture.get("date"),
            "season": league.get("season"),

            "team_id": team.get("id"),
            "team_name": team.get("name"),

            "player_id": player.get("id"),
            "player_name": player.get("name"),

            "absence_type": player.get("type"),
            "absence_reason": player.get("reason")
        })

player_fixture_absences = pd.DataFrame(injury_rows)

player_fixture_absences["fixture_date"] = pd.to_datetime(
    player_fixture_absences["fixture_date"],
    utc=True
)

player_fixture_absences = (
    player_fixture_absences
    .sort_values(
        ["season", "fixture_date", "fixture_id", "team_id", "player_id"]
    )
    .reset_index(drop=True)
)

print("Shape:", player_fixture_absences.shape)

print("\nRows by season:")
print(
    player_fixture_absences
    .groupby("season")
    .size()
)

print("\nAbsence types:")
print(
    player_fixture_absences["absence_type"]
    .value_counts(dropna=False)
)

print("\nMissing IDs:")
print(
    player_fixture_absences[
        ["fixture_id", "team_id", "player_id"]
    ]
    .isna()
    .sum()
)

print("\nDuplicate player-fixture-team rows:")
print(
    player_fixture_absences.duplicated(
        subset=["fixture_id", "team_id", "player_id"]
    ).sum()
)

print("\nSample:")
display(player_fixture_absences.head(10))

In [ ]:
# Only seasons where we have the previous season's player statistics
importance_absences = player_fixture_absences[
    player_fixture_absences["season"].isin([2023, 2024])
].copy()

# The player-stat season we need for each injury season
importance_absences["prior_season"] = (
    importance_absences["season"] - 1
)

# Keep the fields needed from our player-stat table
prior_player_stats = player_season_team[
    [
        "season",
        "player_id",
        "team_id",
        "team_name",
        "minutes",
        "minutes_importance"
    ]
].copy()

prior_player_stats = prior_player_stats.rename(
    columns={
        "season": "prior_season",
        "team_id": "prior_team_id",
        "team_name": "prior_team_name",
        "minutes": "prior_minutes",
        "minutes_importance": "prior_minutes_importance"
    }
)

# Match by player + prior season.
# A player can produce multiple rows here if he played for multiple PL clubs
# in the prior season. We inspect that before choosing the final rule.
absence_prior_matches = importance_absences.merge(
    prior_player_stats,
    on=["player_id", "prior_season"],
    how="left"
)

absence_prior_matches["same_team"] = (
    absence_prior_matches["team_id"]
    == absence_prior_matches["prior_team_id"]
)

print("Absence records before merge:", len(importance_absences))
print("Rows after merge:", len(absence_prior_matches))

print("\nUnique absence records with ANY prior-season PL player match:")
matched_any = (
    absence_prior_matches[
        absence_prior_matches["prior_team_id"].notna()
    ][["fixture_id", "team_id", "player_id"]]
    .drop_duplicates()
)

print(
    len(matched_any),
    "/",
    len(importance_absences),
    f"({len(matched_any) / len(importance_absences):.1%})"
)

print("\nUnique absence records with SAME-TEAM prior-season match:")
matched_same_team = (
    absence_prior_matches[
        absence_prior_matches["same_team"]
    ][["fixture_id", "team_id", "player_id"]]
    .drop_duplicates()
)

print(
    len(matched_same_team),
    "/",
    len(importance_absences),
    f"({len(matched_same_team) / len(importance_absences):.1%})"
)

print("\nCoverage by injury season:")

for season in [2023, 2024]:

    season_absences = importance_absences[
        importance_absences["season"] == season
    ]

    season_keys = season_absences[
        ["fixture_id", "team_id", "player_id"]
    ].drop_duplicates()

    any_keys = matched_any.merge(
        season_keys,
        on=["fixture_id", "team_id", "player_id"],
        how="inner"
    )

    same_keys = matched_same_team.merge(
        season_keys,
        on=["fixture_id", "team_id", "player_id"],
        how="inner"
    )

    print(f"\nSeason {season}")
    print("Absence records:", len(season_keys))
    print(
        "Any prior-team match:",
        len(any_keys),
        f"({len(any_keys) / len(season_keys):.1%})"
    )
    print(
        "Same-team match:",
        len(same_keys),
        f"({len(same_keys) / len(season_keys):.1%})"
    )

In [ ]:
# Count prior-season team records for each player-season
prior_team_counts = (
    prior_player_stats[
        prior_player_stats["prior_team_id"].notna()
    ]
    .groupby(["prior_season", "player_id"])
    .agg(
        prior_team_count=("prior_team_id", "nunique"),
        total_prior_minutes=("prior_minutes", "sum")
    )
    .reset_index()
)

multi_prior_team = prior_team_counts[
    prior_team_counts["prior_team_count"] > 1
].copy()

print("Player-seasons with multiple PL teams:")
print(len(multi_prior_team))

print("\nBy prior season:")
print(
    multi_prior_team
    .groupby("prior_season")
    .size()
)

print("\nTotal prior minutes for multi-team players:")
print(
    multi_prior_team["total_prior_minutes"].describe()
)

# Add names for easier inspection
multi_examples = multi_prior_team.merge(
    player_season_team[
        ["season", "player_id", "player_name"]
    ].drop_duplicates(),
    left_on=["prior_season", "player_id"],
    right_on=["season", "player_id"],
    how="left"
)

print("\nExamples:")
display(
    multi_examples[
        [
            "prior_season",
            "player_id",
            "player_name",
            "prior_team_count",
            "total_prior_minutes"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        ["prior_season", "total_prior_minutes"],
        ascending=[True, False]
    )
    .head(20)
)

In [ ]:
# Start with one row per absence
weighted_absences = importance_absences.copy()

# ---------------------------------------------------------
# 1. SAME-TEAM PRIOR-SEASON MINUTES
# ---------------------------------------------------------

same_team_stats = (
    prior_player_stats
    .sort_values("prior_minutes", ascending=False)
    .drop_duplicates(
        subset=["prior_season", "player_id", "prior_team_id"]
    )
    [
        [
            "prior_season",
            "player_id",
            "prior_team_id",
            "prior_minutes"
        ]
    ]
    .rename(
        columns={
            "prior_team_id": "team_id",
            "prior_minutes": "same_team_prior_minutes"
        }
    )
)

weighted_absences = weighted_absences.merge(
    same_team_stats,
    on=["prior_season", "player_id", "team_id"],
    how="left"
)

# ---------------------------------------------------------
# 2. TRANSFER FALLBACK:
#    maximum prior-season PL minutes from any team entry
# ---------------------------------------------------------

player_prior_max = (
    prior_player_stats
    .groupby(
        ["prior_season", "player_id"],
        as_index=False
    )
    .agg(
        transfer_fallback_minutes=("prior_minutes", "max")
    )
)

weighted_absences = weighted_absences.merge(
    player_prior_max,
    on=["prior_season", "player_id"],
    how="left"
)

# ---------------------------------------------------------
# 3. CHOOSE FINAL PRIOR-SEASON MINUTES
# ---------------------------------------------------------

weighted_absences["importance_minutes"] = (
    weighted_absences["same_team_prior_minutes"]
    .fillna(weighted_absences["transfer_fallback_minutes"])
)

weighted_absences["importance_source"] = "unknown"

weighted_absences.loc[
    weighted_absences["transfer_fallback_minutes"].notna(),
    "importance_source"
] = "transfer_fallback"

weighted_absences.loc[
    weighted_absences["same_team_prior_minutes"].notna(),
    "importance_source"
] = "same_team"

# ---------------------------------------------------------
# 4. CONVERT MINUTES TO 0-1 IMPORTANCE
# ---------------------------------------------------------

weighted_absences["player_importance"] = (
    weighted_absences["importance_minutes"] / MAX_PL_MINUTES
).clip(lower=0, upper=1)

# ---------------------------------------------------------
# CHECK RESULTS
# ---------------------------------------------------------

print("Rows:", len(weighted_absences))

print("\nImportance source:")
print(
    weighted_absences["importance_source"]
    .value_counts()
)

print("\nImportance source percentages:")
print(
    weighted_absences["importance_source"]
    .value_counts(normalize=True)
    .mul(100)
    .round(1)
)

print("\nMissing final importance:")
print(
    weighted_absences["player_importance"]
    .isna()
    .sum()
)

print("\nImportance summary:")
print(
    weighted_absences["player_importance"]
    .describe()
)

In [ ]:
unknown_absences = weighted_absences[
    weighted_absences["importance_source"] == "unknown"
].copy()

print("Total unknown absences:", len(unknown_absences))

print("\nUnknowns by season:")
print(
    unknown_absences["season"]
    .value_counts()
    .sort_index()
)

print("\nUnknowns by absence type:")
print(
    unknown_absences["absence_type"]
    .value_counts()
)

print("\nUnknown percentage within each season:")
for season in [2023, 2024]:

    season_data = weighted_absences[
        weighted_absences["season"] == season
    ]

    unknown_count = (
        season_data["importance_source"] == "unknown"
    ).sum()

    print(
        season,
        f"{unknown_count}/{len(season_data)}",
        f"({unknown_count / len(season_data):.1%})"
    )

print("\nUnknown percentage within each absence type:")
for absence_type in weighted_absences["absence_type"].unique():

    type_data = weighted_absences[
        weighted_absences["absence_type"] == absence_type
    ]

    unknown_count = (
        type_data["importance_source"] == "unknown"
    ).sum()

    print(
        absence_type,
        f"{unknown_count}/{len(type_data)}",
        f"({unknown_count / len(type_data):.1%})"
    )

print("\nSample unknown players:")
display(
    unknown_absences[
        [
            "season",
            "fixture_date",
            "team_name",
            "player_id",
            "player_name",
            "absence_type",
            "absence_reason"
        ]
    ]
    .drop_duplicates("player_id")
    .head(20)
)

In [ ]:
weighted_absences["known_missing_importance"] = 0.0
weighted_absences["known_questionable_importance"] = 0.0

weighted_absences["unknown_missing_player"] = 0
weighted_absences["unknown_questionable_player"] = 0


# Known Missing Fixture importance
mask = (
    (weighted_absences["absence_type"] == "Missing Fixture")
    & weighted_absences["player_importance"].notna()
)

weighted_absences.loc[
    mask,
    "known_missing_importance"
] = weighted_absences.loc[
    mask,
    "player_importance"
]


# Known Questionable importance
mask = (
    (weighted_absences["absence_type"] == "Questionable")
    & weighted_absences["player_importance"].notna()
)

weighted_absences.loc[
    mask,
    "known_questionable_importance"
] = weighted_absences.loc[
    mask,
    "player_importance"
]


# Unknown Missing Fixture players
mask = (
    (weighted_absences["absence_type"] == "Missing Fixture")
    & weighted_absences["player_importance"].isna()
)

weighted_absences.loc[
    mask,
    "unknown_missing_player"
] = 1


# Unknown Questionable players
mask = (
    (weighted_absences["absence_type"] == "Questionable")
    & weighted_absences["player_importance"].isna()
)

weighted_absences.loc[
    mask,
    "unknown_questionable_player"
] = 1


# Aggregate to one row per team per fixture
team_fixture_weighted = (
    weighted_absences
    .groupby(
        [
            "fixture_id",
            "fixture_date",
            "season",
            "team_id",
            "team_name"
        ],
        as_index=False
    )
    .agg(
        missing_importance=(
            "known_missing_importance",
            "sum"
        ),
        questionable_importance=(
            "known_questionable_importance",
            "sum"
        ),
        unknown_missing_players=(
            "unknown_missing_player",
            "sum"
        ),
        unknown_questionable_players=(
            "unknown_questionable_player",
            "sum"
        ),
        total_absence_records=(
            "player_id",
            "count"
        )
    )
)

print("Shape:", team_fixture_weighted.shape)

print("\nRows by season:")
print(
    team_fixture_weighted
    .groupby("season")
    .size()
)

print("\nFeature summary:")
print(
    team_fixture_weighted[
        [
            "missing_importance",
            "questionable_importance",
            "unknown_missing_players",
            "unknown_questionable_players",
            "total_absence_records"
        ]
    ].describe()
)

print("\nSample:")
display(team_fixture_weighted.head(10))

In [ ]:
print("Fixture shape:", fixtures.shape)

print("\nFixture columns:")
for column in fixtures.columns:
    print(column)

print("\nFirst 3 rows:")
display(fixtures.head(3))

In [ ]:
fixtures = pd.read_csv(
    PROCESSED_DIR / "api_football_pl_fixtures.csv"
)

print("Fixture shape:", fixtures.shape)

print("\nFixture columns:")
print(fixtures.columns.tolist())

print("\nFixtures by season:")
print(
    fixtures
    .groupby("api_season")
    .size()
)

print("\nSample:")
display(fixtures.head())

In [ ]:
# Make sure API team IDs use the same data type everywhere

fixtures["home_team_api"] = pd.to_numeric(
    fixtures["home_team_api"],
    errors="coerce"
).astype("Int64")

fixtures["away_team_api"] = pd.to_numeric(
    fixtures["away_team_api"],
    errors="coerce"
).astype("Int64")

team_fixture_weighted["team_id"] = pd.to_numeric(
    team_fixture_weighted["team_id"],
    errors="coerce"
).astype("Int64")

print("Fixture home team ID type:", fixtures["home_team_api"].dtype)
print("Fixture away team ID type:", fixtures["away_team_api"].dtype)
print("Weighted injury team ID type:", team_fixture_weighted["team_id"].dtype)

print("\nMissing converted IDs:")
print("home_team_api:", fixtures["home_team_api"].isna().sum())
print("away_team_api:", fixtures["away_team_api"].isna().sum())
print("team_id:", team_fixture_weighted["team_id"].isna().sum())

In [ ]:
weighted_fixture_features = fixtures[
    [
        "api_fixture_id",
        "date",
        "api_season",
        "home_team_api",
        "away_team_api",
        "home_team",
        "away_team"
    ]
].copy()

# -----------------------------
# HOME TEAM FEATURES
# -----------------------------

home_weighted = team_fixture_weighted.rename(
    columns={
        "fixture_id": "api_fixture_id",
        "season": "api_season",
        "team_id": "home_team_api",
        "missing_importance": "missing_importance_home",
        "questionable_importance": "questionable_importance_home",
        "unknown_missing_players": "unknown_missing_players_home",
        "unknown_questionable_players": "unknown_questionable_players_home",
        "total_absence_records": "total_absence_records_home"
    }
)

home_weighted = home_weighted[
    [
        "api_fixture_id",
        "api_season",
        "home_team_api",
        "missing_importance_home",
        "questionable_importance_home",
        "unknown_missing_players_home",
        "unknown_questionable_players_home",
        "total_absence_records_home"
    ]
]

weighted_fixture_features = weighted_fixture_features.merge(
    home_weighted,
    on=[
        "api_fixture_id",
        "api_season",
        "home_team_api"
    ],
    how="left"
)

# -----------------------------
# AWAY TEAM FEATURES
# -----------------------------

away_weighted = team_fixture_weighted.rename(
    columns={
        "fixture_id": "api_fixture_id",
        "season": "api_season",
        "team_id": "away_team_api",
        "missing_importance": "missing_importance_away",
        "questionable_importance": "questionable_importance_away",
        "unknown_missing_players": "unknown_missing_players_away",
        "unknown_questionable_players": "unknown_questionable_players_away",
        "total_absence_records": "total_absence_records_away"
    }
)

away_weighted = away_weighted[
    [
        "api_fixture_id",
        "api_season",
        "away_team_api",
        "missing_importance_away",
        "questionable_importance_away",
        "unknown_missing_players_away",
        "unknown_questionable_players_away",
        "total_absence_records_away"
    ]
]

weighted_fixture_features = weighted_fixture_features.merge(
    away_weighted,
    on=[
        "api_fixture_id",
        "api_season",
        "away_team_api"
    ],
    how="left"
)

# -----------------------------
# FILL NO-ABSENCE TEAM-FIXTURES WITH ZERO
# -----------------------------

weighted_cols = [
    "missing_importance_home",
    "questionable_importance_home",
    "unknown_missing_players_home",
    "unknown_questionable_players_home",
    "total_absence_records_home",
    "missing_importance_away",
    "questionable_importance_away",
    "unknown_missing_players_away",
    "unknown_questionable_players_away",
    "total_absence_records_away"
]

weighted_fixture_features[weighted_cols] = (
    weighted_fixture_features[weighted_cols]
    .fillna(0)
)

print("Shape:", weighted_fixture_features.shape)

print("\nRows by season:")
print(
    weighted_fixture_features
    .groupby("api_season")
    .size()
)

print("\nMissing values in weighted features:")
print(
    weighted_fixture_features[weighted_cols]
    .isna()
    .sum()
)

print("\nSample:")
display(weighted_fixture_features.head(10))

In [ ]:
weighted_fixture_features["missing_importance_difference"] = (
    weighted_fixture_features["missing_importance_home"]
    - weighted_fixture_features["missing_importance_away"]
)

weighted_fixture_features["questionable_importance_difference"] = (
    weighted_fixture_features["questionable_importance_home"]
    - weighted_fixture_features["questionable_importance_away"]
)

weighted_fixture_features["unknown_missing_players_difference"] = (
    weighted_fixture_features["unknown_missing_players_home"]
    - weighted_fixture_features["unknown_missing_players_away"]
)

weighted_fixture_features["unknown_questionable_players_difference"] = (
    weighted_fixture_features["unknown_questionable_players_home"]
    - weighted_fixture_features["unknown_questionable_players_away"]
)

difference_cols = [
    "missing_importance_difference",
    "questionable_importance_difference",
    "unknown_missing_players_difference",
    "unknown_questionable_players_difference"
]

print("Shape:", weighted_fixture_features.shape)

print("\nDifference feature summary:")
print(
    weighted_fixture_features[difference_cols]
    .describe()
)

print("\nMissing values:")
print(
    weighted_fixture_features[difference_cols]
    .isna()
    .sum()
)

print("\nSample:")
display(
    weighted_fixture_features[
        [
            "api_season",
            "home_team",
            "away_team",
            "missing_importance_home",
            "missing_importance_away",
            "missing_importance_difference",
            "unknown_missing_players_home",
            "unknown_missing_players_away",
            "unknown_missing_players_difference"
        ]
    ].head(10)
)

In [ ]:
print("=== WEIGHTED INJURY TABLE ===")
print("Rows:", len(team_fixture_weighted))
print("Seasons:", sorted(team_fixture_weighted["season"].unique()))
print("Unique fixture IDs:", team_fixture_weighted["fixture_id"].nunique())

print("\n=== COMPLETE FIXTURE TABLE ===")
print("Rows:", len(fixtures))
print("Seasons:", sorted(fixtures["api_season"].unique()))
print("Unique fixture IDs:", fixtures["api_fixture_id"].nunique())

weighted_ids = set(
    pd.to_numeric(
        team_fixture_weighted["fixture_id"],
        errors="coerce"
    ).dropna().astype(int)
)

fixture_ids = set(
    pd.to_numeric(
        fixtures["api_fixture_id"],
        errors="coerce"
    ).dropna().astype(int)
)

overlapping_ids = weighted_ids.intersection(fixture_ids)

print("\n=== FIXTURE ID OVERLAP ===")
print("Weighted fixture IDs:", len(weighted_ids))
print("Complete fixture IDs:", len(fixture_ids))
print("Overlapping fixture IDs:", len(overlapping_ids))

print("\n=== CURRENT MERGE CHECK ===")

check_cols = [
    "missing_importance_home",
    "missing_importance_away",
    "questionable_importance_home",
    "questionable_importance_away",
    "total_absence_records_home",
    "total_absence_records_away"
]

for col in check_cols:
    print(
        col,
        "| non-zero rows:",
        (weighted_fixture_features[col] != 0).sum(),
        "| max:",
        weighted_fixture_features[col].max()
    )

print("\n=== SAMPLE WEIGHTED INJURY ROWS ===")
display(
    team_fixture_weighted[
        [
            "fixture_id",
            "season",
            "team_id",
            "team_name",
            "missing_importance",
            "questionable_importance",
            "total_absence_records"
        ]
    ].head(10)
)

In [ ]:
team_id_check = team_fixture_weighted[
    [
        "fixture_id",
        "season",
        "team_id",
        "team_name"
    ]
].merge(
    fixtures[
        [
            "api_fixture_id",
            "api_season",
            "home_team_api",
            "away_team_api",
            "home_team",
            "away_team"
        ]
    ],
    left_on="fixture_id",
    right_on="api_fixture_id",
    how="left"
)

team_id_check["matches_home_id"] = (
    team_id_check["team_id"] == team_id_check["home_team_api"]
)

team_id_check["matches_away_id"] = (
    team_id_check["team_id"] == team_id_check["away_team_api"]
)

team_id_check["matches_either_id"] = (
    team_id_check["matches_home_id"]
    | team_id_check["matches_away_id"]
)

print("Rows checked:", len(team_id_check))

print("\nTeam ID matches:")
print(team_id_check[
    [
        "matches_home_id",
        "matches_away_id",
        "matches_either_id"
    ]
].sum())

print("\nRows where injury team ID matches neither home nor away:")
print((~team_id_check["matches_either_id"]).sum())

print("\nSample comparison:")
display(
    team_id_check[
        [
            "fixture_id",
            "season",
            "team_id",
            "team_name",
            "home_team_api",
            "home_team",
            "away_team_api",
            "away_team",
            "matches_home_id",
            "matches_away_id"
        ]
    ].head(15)
)

In [ ]:
# Standardize fixture IDs in both tables

fixtures["api_fixture_id"] = pd.to_numeric(
    fixtures["api_fixture_id"],
    errors="coerce"
).astype("Int64")

team_fixture_weighted["fixture_id"] = pd.to_numeric(
    team_fixture_weighted["fixture_id"],
    errors="coerce"
).astype("Int64")

print("Fixture table fixture ID type:", fixtures["api_fixture_id"].dtype)
print("Weighted injury fixture ID type:", team_fixture_weighted["fixture_id"].dtype)

print("\nMissing fixture IDs after conversion:")
print(
    "fixtures:",
    fixtures["api_fixture_id"].isna().sum()
)
print(
    "team_fixture_weighted:",
    team_fixture_weighted["fixture_id"].isna().sum()
)

print("\nUnique fixture IDs:")
print(
    "fixtures:",
    fixtures["api_fixture_id"].nunique()
)
print(
    "team_fixture_weighted:",
    team_fixture_weighted["fixture_id"].nunique()
)

In [ ]:
team_id_check = team_fixture_weighted[
    [
        "fixture_id",
        "season",
        "team_id",
        "team_name"
    ]
].merge(
    fixtures[
        [
            "api_fixture_id",
            "api_season",
            "home_team_api",
            "away_team_api",
            "home_team",
            "away_team"
        ]
    ],
    left_on="fixture_id",
    right_on="api_fixture_id",
    how="left"
)

team_id_check["matches_home_id"] = (
    team_id_check["team_id"] == team_id_check["home_team_api"]
)

team_id_check["matches_away_id"] = (
    team_id_check["team_id"] == team_id_check["away_team_api"]
)

team_id_check["matches_either_id"] = (
    team_id_check["matches_home_id"]
    | team_id_check["matches_away_id"]
)

print("Rows checked:", len(team_id_check))

print("\nFixture rows successfully joined:")
print(team_id_check["api_fixture_id"].notna().sum())

print("\nTeam ID matches:")
print(
    team_id_check[
        [
            "matches_home_id",
            "matches_away_id",
            "matches_either_id"
        ]
    ].sum()
)

print("\nRows with a definite False for matches_either_id:")
print(
    (team_id_check["matches_either_id"] == False).sum()
)

print("\nMissing match comparisons:")
print(
    team_id_check["matches_either_id"].isna().sum()
)

print("\nSample:")
display(
    team_id_check[
        [
            "fixture_id",
            "team_id",
            "team_name",
            "home_team_api",
            "home_team",
            "away_team_api",
            "away_team",
            "matches_home_id",
            "matches_away_id",
            "matches_either_id"
        ]
    ].head(15)
)

In [ ]:
print("=== FIXTURE TEAM ID COLUMNS ===")

print("\nhome_team_api dtype:")
print(fixtures["home_team_api"].dtype)

print("\naway_team_api dtype:")
print(fixtures["away_team_api"].dtype)

print("\nMissing values:")
print("home_team_api:", fixtures["home_team_api"].isna().sum())
print("away_team_api:", fixtures["away_team_api"].isna().sum())

print("\nUnique values:")
print("home_team_api:", fixtures["home_team_api"].nunique())
print("away_team_api:", fixtures["away_team_api"].nunique())

print("\nFirst 15 fixture team IDs:")
display(
    fixtures[
        [
            "api_fixture_id",
            "home_team_api",
            "home_team",
            "away_team_api",
            "away_team"
        ]
    ].head(15)
)

print("\n=== WEIGHTED INJURY TEAM IDS ===")

print("team_id dtype:", team_fixture_weighted["team_id"].dtype)
print("Missing team_id:", team_fixture_weighted["team_id"].isna().sum())
print("Unique team IDs:", team_fixture_weighted["team_id"].nunique())

display(
    team_fixture_weighted[
        [
            "fixture_id",
            "team_id",
            "team_name"
        ]
    ].head(15)
)

In [ ]:
team_name_check = team_fixture_weighted[
    [
        "fixture_id",
        "season",
        "team_name"
    ]
].merge(
    fixtures[
        [
            "api_fixture_id",
            "api_season",
            "home_team",
            "away_team"
        ]
    ],
    left_on="fixture_id",
    right_on="api_fixture_id",
    how="left"
)

team_name_check["matches_home_name"] = (
    team_name_check["team_name"].str.strip().str.lower()
    == team_name_check["home_team"].str.strip().str.lower()
)

team_name_check["matches_away_name"] = (
    team_name_check["team_name"].str.strip().str.lower()
    == team_name_check["away_team"].str.strip().str.lower()
)

team_name_check["matches_either_name"] = (
    team_name_check["matches_home_name"]
    | team_name_check["matches_away_name"]
)

print("Rows checked:", len(team_name_check))

print("\nTeam-name matches:")
print(
    team_name_check[
        [
            "matches_home_name",
            "matches_away_name",
            "matches_either_name"
        ]
    ].sum()
)

print("\nRows matching neither home nor away:")
print(
    (~team_name_check["matches_either_name"]).sum()
)

print("\nNon-matching examples:")
display(
    team_name_check.loc[
        ~team_name_check["matches_either_name"],
        [
            "fixture_id",
            "team_name",
            "home_team",
            "away_team"
        ]
    ].head(25)
)

In [ ]:
non_matching = team_name_check.loc[
    ~team_name_check["matches_either_name"],
    [
        "fixture_id",
        "season",
        "team_name",
        "home_team",
        "away_team"
    ]
].copy()

print("Non-matching rows:", len(non_matching))

print("\nUnique injury-side team names in mismatches:")
print(
    non_matching["team_name"]
    .value_counts()
    .head(30)
)

print("\nUnique fixture home-team names in mismatches:")
print(
    non_matching["home_team"]
    .value_counts()
    .head(30)
)

print("\nUnique fixture away-team names in mismatches:")
print(
    non_matching["away_team"]
    .value_counts()
    .head(30)
)

print("\nSample mismatches:")
display(non_matching.head(40))

In [ ]:
team_aliases = {
    "Brighton": "Brighton & Hove Albion",
    "Newcastle": "Newcastle United",
    "Tottenham": "Tottenham Hotspur",
    "Wolves": "Wolverhampton",
    "West Ham": "West Ham United",
    "Luton": "Luton Town",
    "Sheffield Utd": "Sheffield United",
    "Ipswich": "Ipswich Town",
    "Leicester": "Leicester City"
}

team_name_check["normalized_team_name"] = (
    team_name_check["team_name"]
    .replace(team_aliases)
    .str.strip()
    .str.lower()
)

team_name_check["normalized_home_team"] = (
    team_name_check["home_team"]
    .str.strip()
    .str.lower()
)

team_name_check["normalized_away_team"] = (
    team_name_check["away_team"]
    .str.strip()
    .str.lower()
)

team_name_check["matches_home_normalized"] = (
    team_name_check["normalized_team_name"]
    == team_name_check["normalized_home_team"]
)

team_name_check["matches_away_normalized"] = (
    team_name_check["normalized_team_name"]
    == team_name_check["normalized_away_team"]
)

team_name_check["matches_either_normalized"] = (
    team_name_check["matches_home_normalized"]
    | team_name_check["matches_away_normalized"]
)

print("Rows checked:", len(team_name_check))

print("\nNormalized matches:")
print(
    team_name_check[
        [
            "matches_home_normalized",
            "matches_away_normalized",
            "matches_either_normalized"
        ]
    ].sum()
)

print("\nRows matching neither after normalization:")
print(
    (~team_name_check["matches_either_normalized"]).sum()
)

print("\nRows matching BOTH home and away:")
print(
    (
        team_name_check["matches_home_normalized"]
        & team_name_check["matches_away_normalized"]
    ).sum()
)

print("\nRemaining mismatches:")
display(
    team_name_check.loc[
        ~team_name_check["matches_either_normalized"],
        [
            "fixture_id",
            "season",
            "team_name",
            "home_team",
            "away_team"
        ]
    ].head(20)
)

In [ ]:
# --------------------------------------------------
# STEP 24: REBUILD WEIGHTED FIXTURE FEATURES CORRECTLY
# --------------------------------------------------
import numpy as np

weighted_for_merge = team_fixture_weighted.copy()

# Normalize injury-side team names
weighted_for_merge["normalized_team_name"] = (
    weighted_for_merge["team_name"]
    .replace(team_aliases)
    .str.strip()
    .str.lower()
)

# Start from the complete fixture table
weighted_fixture_features = fixtures[
    [
        "api_fixture_id",
        "date",
        "api_season",
        "home_team",
        "away_team"
    ]
].copy()

# Normalize fixture team names
weighted_fixture_features["normalized_home_team"] = (
    weighted_fixture_features["home_team"]
    .str.strip()
    .str.lower()
)

weighted_fixture_features["normalized_away_team"] = (
    weighted_fixture_features["away_team"]
    .str.strip()
    .str.lower()
)

# Attach fixture home/away names to each injury-team row
weighted_for_merge = weighted_for_merge.merge(
    weighted_fixture_features[
        [
            "api_fixture_id",
            "normalized_home_team",
            "normalized_away_team"
        ]
    ],
    left_on="fixture_id",
    right_on="api_fixture_id",
    how="left",
    validate="many_to_one"
)

# Determine whether each injury record belongs to home or away
weighted_for_merge["side"] = np.select(
    [
        weighted_for_merge["normalized_team_name"]
        == weighted_for_merge["normalized_home_team"],

        weighted_for_merge["normalized_team_name"]
        == weighted_for_merge["normalized_away_team"]
    ],
    [
        "home",
        "away"
    ],
    default="unmatched"
)

print("Side assignment:")
print(weighted_for_merge["side"].value_counts(dropna=False))

# --------------------------------------------------
# HOME FEATURES
# --------------------------------------------------

home_weighted = (
    weighted_for_merge[
        weighted_for_merge["side"] == "home"
    ][
        [
            "fixture_id",
            "missing_importance",
            "questionable_importance",
            "unknown_missing_players",
            "unknown_questionable_players",
            "total_absence_records"
        ]
    ]
    .rename(
        columns={
            "fixture_id": "api_fixture_id",
            "missing_importance": "missing_importance_home",
            "questionable_importance": "questionable_importance_home",
            "unknown_missing_players": "unknown_missing_players_home",
            "unknown_questionable_players": "unknown_questionable_players_home",
            "total_absence_records": "total_absence_records_home"
        }
    )
)

# --------------------------------------------------
# AWAY FEATURES
# --------------------------------------------------

away_weighted = (
    weighted_for_merge[
        weighted_for_merge["side"] == "away"
    ][
        [
            "fixture_id",
            "missing_importance",
            "questionable_importance",
            "unknown_missing_players",
            "unknown_questionable_players",
            "total_absence_records"
        ]
    ]
    .rename(
        columns={
            "fixture_id": "api_fixture_id",
            "missing_importance": "missing_importance_away",
            "questionable_importance": "questionable_importance_away",
            "unknown_missing_players": "unknown_missing_players_away",
            "unknown_questionable_players": "unknown_questionable_players_away",
            "total_absence_records": "total_absence_records_away"
        }
    )
)

# Merge home and away features onto all 1,140 fixtures
weighted_fixture_features = weighted_fixture_features.merge(
    home_weighted,
    on="api_fixture_id",
    how="left",
    validate="one_to_one"
)

weighted_fixture_features = weighted_fixture_features.merge(
    away_weighted,
    on="api_fixture_id",
    how="left",
    validate="one_to_one"
)

# A missing team-fixture injury row means no recorded absences
weighted_cols = [
    "missing_importance_home",
    "questionable_importance_home",
    "unknown_missing_players_home",
    "unknown_questionable_players_home",
    "total_absence_records_home",
    "missing_importance_away",
    "questionable_importance_away",
    "unknown_missing_players_away",
    "unknown_questionable_players_away",
    "total_absence_records_away"
]

weighted_fixture_features[weighted_cols] = (
    weighted_fixture_features[weighted_cols]
    .fillna(0)
)

# Create home - away differences
weighted_fixture_features["missing_importance_difference"] = (
    weighted_fixture_features["missing_importance_home"]
    - weighted_fixture_features["missing_importance_away"]
)

weighted_fixture_features["questionable_importance_difference"] = (
    weighted_fixture_features["questionable_importance_home"]
    - weighted_fixture_features["questionable_importance_away"]
)

weighted_fixture_features["unknown_missing_players_difference"] = (
    weighted_fixture_features["unknown_missing_players_home"]
    - weighted_fixture_features["unknown_missing_players_away"]
)

weighted_fixture_features["unknown_questionable_players_difference"] = (
    weighted_fixture_features["unknown_questionable_players_home"]
    - weighted_fixture_features["unknown_questionable_players_away"]
)

difference_cols = [
    "missing_importance_difference",
    "questionable_importance_difference",
    "unknown_missing_players_difference",
    "unknown_questionable_players_difference"
]

print("\nFinal shape:", weighted_fixture_features.shape)

print("\nRows by season:")
print(
    weighted_fixture_features
    .groupby("api_season")
    .size()
)

print("\nNon-zero weighted features:")
for col in weighted_cols:
    print(
        col,
        "| non-zero:",
        (weighted_fixture_features[col] != 0).sum(),
        "| max:",
        weighted_fixture_features[col].max()
    )

print("\nDifference summary:")
print(
    weighted_fixture_features[difference_cols]
    .describe()
)

print("\nTotal missing weighted values:")
print(
    weighted_fixture_features[
        weighted_cols + difference_cols
    ]
    .isna()
    .sum()
    .sum()
)

In [ ]:
weighted_output_path = (
    PROCESSED_DIR / "api_football_player_weighted_injury_features.csv"
)

weighted_fixture_features.to_csv(
    weighted_output_path,
    index=False
)

print("Saved to:")
print(weighted_output_path)

print("\nSaved shape:")
print(weighted_fixture_features.shape)

print("\nSeasons:")
print(
    weighted_fixture_features
    .groupby("api_season")
    .size()
)

print("\nFile exists:")
print(weighted_output_path.exists())

In [ ]:
injury_model_df = pd.read_csv(
    PROCESSED_DIR / "pl_match_features_with_injuries_experimental.csv"
)

print("Shape:", injury_model_df.shape)

print("\nColumns:")
for col in injury_model_df.columns:
    print(col)

print("\nSeason counts:")
if "season" in injury_model_df.columns:
    print(injury_model_df.groupby("season").size())

print("\nFirst 3 rows:")
display(injury_model_df.head(3))

In [ ]:
weighted_model_features = weighted_fixture_features[
    [
        "api_fixture_id",
        "api_season",
        "missing_importance_home",
        "questionable_importance_home",
        "unknown_missing_players_home",
        "unknown_questionable_players_home",
        "missing_importance_away",
        "questionable_importance_away",
        "unknown_missing_players_away",
        "unknown_questionable_players_away",
        "missing_importance_difference",
        "questionable_importance_difference",
        "unknown_missing_players_difference",
        "unknown_questionable_players_difference"
    ]
].copy()

weighted_model_features = weighted_model_features[
    weighted_model_features["api_season"].isin([2023, 2024])
].copy()

weighted_model_df = injury_model_df[
    injury_model_df["season"].isin(["2023_24", "2024_25"])
].copy()

weighted_model_df = weighted_model_df.merge(
    weighted_model_features,
    on="api_fixture_id",
    how="left",
    validate="one_to_one"
)

weighted_feature_cols = [
    "missing_importance_home",
    "questionable_importance_home",
    "unknown_missing_players_home",
    "unknown_questionable_players_home",
    "missing_importance_away",
    "questionable_importance_away",
    "unknown_missing_players_away",
    "unknown_questionable_players_away",
    "missing_importance_difference",
    "questionable_importance_difference",
    "unknown_missing_players_difference",
    "unknown_questionable_players_difference"
]

print("Shape:", weighted_model_df.shape)

print("\nRows by season:")
print(
    weighted_model_df
    .groupby("season")
    .size()
)

print("\nMissing weighted features:")
print(
    weighted_model_df[weighted_feature_cols]
    .isna()
    .sum()
)

print("\nRows with any missing weighted feature:")
print(
    weighted_model_df[weighted_feature_cols]
    .isna()
    .any(axis=1)
    .sum()
)

print("\nSample:")
display(
    weighted_model_df[
        [
            "season",
            "home_team",
            "away_team",
            "missing_importance_home",
            "missing_importance_away",
            "missing_importance_difference",
            "unknown_missing_players_home",
            "unknown_missing_players_away"
        ]
    ].head(10)
)

In [ ]:
baseline_features = [
    "points_last5_all_home",
    "goal_difference_last5_all_home",
    "win_rate_last5_all_home",
    "shots_on_target_difference_last5_all_home",
    "points_last5_all_away",
    "goal_difference_last5_all_away",
    "win_rate_last5_all_away",
    "shots_on_target_difference_last5_all_away",

    "points_last5_pl_home",
    "goal_difference_last5_pl_home",
    "win_rate_last5_pl_home",
    "shots_on_target_difference_last5_pl_home",
    "points_last10_pl_home",
    "goal_difference_last10_pl_home",
    "win_rate_last10_pl_home",

    "points_last5_pl_away",
    "goal_difference_last5_pl_away",
    "win_rate_last5_pl_away",
    "shots_on_target_difference_last5_pl_away",
    "points_last10_pl_away",
    "goal_difference_last10_pl_away",
    "win_rate_last10_pl_away",

    "days_rest_home",
    "days_rest_away",

    "matches_last_7_days_home",
    "matches_last_7_days_away",
    "matches_last_14_days_home",
    "matches_last_14_days_away",

    "team_elo_home",
    "team_elo_away",
    "team_elo_difference_home"
]

print("Baseline feature count:", len(baseline_features))

missing_columns = [
    col
    for col in baseline_features
    if col not in weighted_model_df.columns
]

print("Baseline columns missing from weighted_model_df:")
print(missing_columns)

In [ ]:
v1_features = baseline_features.copy()

v2_injury_features = [
    "missing_fixture_home",
    "questionable_home",
    "missing_fixture_away",
    "questionable_away",
    "missing_fixture_difference",
    "questionable_difference"
]

v4_weighted_injury_features = [
    "missing_importance_home",
    "questionable_importance_home",
    "missing_importance_away",
    "questionable_importance_away",
    "missing_importance_difference",
    "questionable_importance_difference",
    "unknown_missing_players_home",
    "unknown_missing_players_away"
]

v2_features = v1_features + v2_injury_features
v4_features = v1_features + v4_weighted_injury_features

print("V1 feature count:", len(v1_features))
print("V2 feature count:", len(v2_features))
print("V4 feature count:", len(v4_features))

print("\nMissing values:")
print("V1:", weighted_model_df[v1_features].isna().sum().sum())
print("V2:", weighted_model_df[v2_features].isna().sum().sum())
print("V4:", weighted_model_df[v4_features].isna().sum().sum())

In [ ]:
# TEST V1

train_df = weighted_model_df[
    weighted_model_df["season"] == "2023_24"
].copy()

test_df = weighted_model_df[
    weighted_model_df["season"] == "2024_25"
].copy()

print("Training matches:", len(train_df))
print("Training season:")
print(train_df["season"].value_counts())

print("\nTest matches:", len(test_df))
print("Test season:")
print(test_df["season"].value_counts())

print("\nTarget distribution - training:")
print(
    train_df["target"]
    .value_counts()
    .sort_index()
)

print("\nTarget distribution - test:")
print(
    test_df["target"]
    .value_counts()
    .sort_index()
)

In [ ]:
# Evaluate all models

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, log_loss
import pandas as pd


def evaluate_model_version(name, feature_cols, train_df, test_df):
    
    # Separate features and target
    X_train = train_df[feature_cols].copy()
    y_train = train_df["target"].copy()

    X_test = test_df[feature_cols].copy()
    y_test = test_df["target"].copy()

    # Fit imputer ONLY on training data
    imputer = SimpleImputer(
        strategy="median",
        add_indicator=True
    )

    X_train_imputed = imputer.fit_transform(X_train)
    X_test_imputed = imputer.transform(X_test)

    # Same locked Random Forest configuration
    model = RandomForestClassifier(
        n_estimators=500,
        max_depth=6,
        min_samples_leaf=10,
        max_features="sqrt",
        criterion="gini",
        class_weight=None,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train_imputed, y_train)

    # Predictions
    predictions = model.predict(X_test_imputed)
    probabilities = model.predict_proba(X_test_imputed)

    accuracy = accuracy_score(
        y_test,
        predictions
    )

    loss = log_loss(
        y_test,
        probabilities,
        labels=model.classes_
    )

    return {
        "Model": name,
        "Features": len(feature_cols),
        "Train Matches": len(train_df),
        "Test Matches": len(test_df),
        "Accuracy": accuracy,
        "Log Loss": loss
    }


results = []

results.append(
    evaluate_model_version(
        "V1 - Baseline",
        v1_features,
        train_df,
        test_df
    )
)

results.append(
    evaluate_model_version(
        "V2 - Raw Injuries",
        v2_features,
        train_df,
        test_df
    )
)

results.append(
    evaluate_model_version(
        "V4 - Player Weighted Injuries",
        v4_features,
        train_df,
        test_df
    )
)

comparison_df = pd.DataFrame(results)

print("2024/25 OUT-OF-SAMPLE COMPARISON")
print("--------------------------------")
display(comparison_df)

print("\nBest accuracy:")
print(
    comparison_df.loc[
        comparison_df["Accuracy"].idxmax(),
        ["Model", "Accuracy"]
    ]
)

print("\nBest log loss:")
print(
    comparison_df.loc[
        comparison_df["Log Loss"].idxmin(),
        ["Model", "Log Loss"]
    ]
)

In [ ]:
v1_accuracy = comparison_df.loc[
    comparison_df["Model"] == "V1 - Baseline",
    "Accuracy"
].iloc[0]

v1_log_loss = comparison_df.loc[
    comparison_df["Model"] == "V1 - Baseline",
    "Log Loss"
].iloc[0]


comparison_df["Accuracy Change vs V1"] = (
    comparison_df["Accuracy"] - v1_accuracy
)

comparison_df["Log Loss Change vs V1"] = (
    comparison_df["Log Loss"] - v1_log_loss
)

comparison_df["Log Loss Improvement %"] = (
    (v1_log_loss - comparison_df["Log Loss"])
    / v1_log_loss
) * 100


display(
    comparison_df[
        [
            "Model",
            "Accuracy",
            "Accuracy Change vs V1",
            "Log Loss",
            "Log Loss Change vs V1",
            "Log Loss Improvement %"
        ]
    ]
)

In [ ]:
v4_output_path = (
    PROCESSED_DIR / "pl_match_features_player_weighted_injuries_v4.csv"
)

weighted_model_df.to_csv(
    v4_output_path,
    index=False
)

print("Saved to:")
print(v4_output_path)

print("\nShape:")
print(weighted_model_df.shape)

print("\nSeasons:")
print(weighted_model_df.groupby("season").size())

print("\nFile exists:")
print(v4_output_path.exists())

In [ ]:
import pandas as pd
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")

historical_df = pd.read_csv(
    PROCESSED_DIR / "historical_matches.csv"
)

print("Shape:", historical_df.shape)

print("\nColumns containing odds-related terms:")
odds_columns = [
    col for col in historical_df.columns
    if any(
        term in col.lower()
        for term in ["odd", "bet365", "b365"]
    )
]

print(odds_columns)

print("\nAll columns:")
for col in historical_df.columns:
    print(col)

In [ ]:
import json
from pathlib import Path

RAW_MATCH_DIR = Path("../data/raw/matches")

sample_season_dir = (
    RAW_MATCH_DIR
    / "premier_league"
    / "2025_26"
)

json_files = sorted(sample_season_dir.glob("*.json"))

print("JSON files found:", len(json_files))

sample_file = json_files[0]

print("\nSample file:")
print(sample_file.name)

with open(sample_file, "r", encoding="utf-8") as f:
    sample_raw = json.load(f)

print("\nTop-level keys:")
print(sample_raw.keys())

print("\nOdds object:")
print(
    json.dumps(
        sample_raw.get("odds"),
        indent=2
    )[:5000]
)

In [ ]:
import json
import pandas as pd
from pathlib import Path

RAW_MATCH_DIR = Path("../data/raw/matches")

pl_dir = RAW_MATCH_DIR / "premier_league"

odds_rows = []

for season_dir in sorted(pl_dir.iterdir()):

    if not season_dir.is_dir():
        continue

    season = season_dir.name

    for file_path in sorted(season_dir.glob("*.json")):

        with open(file_path, "r", encoding="utf-8") as f:
            raw = json.load(f)

        odds = raw.get("odds")

        if not odds:
            continue

        odds_data = odds.get("data")

        if not odds_data:
            continue

        bookmakers = odds_data.get("bookmakers", [])

        bet365 = next(
            (
                bookmaker
                for bookmaker in bookmakers
                if bookmaker.get("bookmaker") == "Bet365"
            ),
            None
        )

        if bet365 is None:
            continue

        match_odds = (
            bet365
            .get("markets", {})
            .get("match_odds", {})
        )

        home_odds = (
            match_odds
            .get("home", {})
            .get("last_seen")
        )

        draw_odds = (
            match_odds
            .get("draw", {})
            .get("last_seen")
        )

        away_odds = (
            match_odds
            .get("away", {})
            .get("last_seen")
        )

        odds_rows.append({
            "match_id": odds_data.get("match_id"),
            "season": season,
            "bet365_home_odds": pd.to_numeric(
                home_odds,
                errors="coerce"
            ),
            "bet365_draw_odds": pd.to_numeric(
                draw_odds,
                errors="coerce"
            ),
            "bet365_away_odds": pd.to_numeric(
                away_odds,
                errors="coerce"
            )
        })


bet365_odds_df = pd.DataFrame(odds_rows)

print("Extracted rows:", len(bet365_odds_df))

print("\nRows by season:")
print(
    bet365_odds_df
    .groupby("season")
    .size()
)

print("\nMissing odds:")
print(
    bet365_odds_df[
        [
            "bet365_home_odds",
            "bet365_draw_odds",
            "bet365_away_odds"
        ]
    ]
    .isna()
    .sum()
)

print("\nSample:")
display(bet365_odds_df.head())

In [ ]:
# Raw implied probabilities
bet365_odds_df["market_home_raw"] = (
    1 / bet365_odds_df["bet365_home_odds"]
)

bet365_odds_df["market_draw_raw"] = (
    1 / bet365_odds_df["bet365_draw_odds"]
)

bet365_odds_df["market_away_raw"] = (
    1 / bet365_odds_df["bet365_away_odds"]
)


# Bookmaker overround
bet365_odds_df["market_overround"] = (
    bet365_odds_df["market_home_raw"]
    + bet365_odds_df["market_draw_raw"]
    + bet365_odds_df["market_away_raw"]
)


# Normalize to remove bookmaker margin
bet365_odds_df["market_home_fair"] = (
    bet365_odds_df["market_home_raw"]
    / bet365_odds_df["market_overround"]
)

bet365_odds_df["market_draw_fair"] = (
    bet365_odds_df["market_draw_raw"]
    / bet365_odds_df["market_overround"]
)

bet365_odds_df["market_away_fair"] = (
    bet365_odds_df["market_away_raw"]
    / bet365_odds_df["market_overround"]
)


# Check that fair probabilities sum to 1
bet365_odds_df["fair_probability_sum"] = (
    bet365_odds_df["market_home_fair"]
    + bet365_odds_df["market_draw_fair"]
    + bet365_odds_df["market_away_fair"]
)


print("Average bookmaker overround:")
print(bet365_odds_df["market_overround"].mean())

print("\nAverage bookmaker margin:")
print(
    (bet365_odds_df["market_overround"].mean() - 1) * 100,
    "%"
)

print("\nFair probability sum:")
print(
    bet365_odds_df["fair_probability_sum"].describe()
)

print("\nSample:")
display(
    bet365_odds_df[
        [
            "match_id",
            "bet365_home_odds",
            "bet365_draw_odds",
            "bet365_away_odds",
            "market_overround",
            "market_home_fair",
            "market_draw_fair",
            "market_away_fair",
            "fair_probability_sum"
        ]
    ].head()
)

In [ ]:
market_output_path = (
    PROCESSED_DIR / "bet365_pl_market_probabilities.csv"
)

market_columns = [
    "match_id",
    "season",
    "bet365_home_odds",
    "bet365_draw_odds",
    "bet365_away_odds",
    "market_overround",
    "market_home_fair",
    "market_draw_fair",
    "market_away_fair"
]

bet365_market_df = bet365_odds_df[
    market_columns
].copy()

bet365_market_df.to_csv(
    market_output_path,
    index=False
)

print("Saved to:")
print(market_output_path)

print("\nShape:")
print(bet365_market_df.shape)

print("\nRows by season:")
print(
    bet365_market_df
    .groupby("season")
    .size()
)

print("\nMissing values:")
print(
    bet365_market_df
    .isna()
    .sum()
)

print("\nFile exists:")
print(market_output_path.exists())